# 12 — V2 evaluation and error analysis: decision evidence, feature additions, slices, calibration, drift

**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED.** ไม่ใช่ Dashboard/เว็บ หรือการประเมินความเสี่ยงรายเส้นทาง/รายบุคคล

- อินพุต (อ่านอย่างเดียว ตรวจ hash ก่อนอ่าน prediction): ผล notebook 11 ที่ `status == completed` (`reports/v2/models/model_manifest.json`, `selection_lock.json`,
  ตาราง metric, `data/v2/models/*_predictions_*.csv.gz`), panel/feature ของ 10 ที่ตรวจรับแล้ว (context สำหรับ slice เท่านั้น), `workflow/model-protocol-v2-lock.json`
- ไม่มีเครือข่าย, ไม่ fit, ไม่ tune/calibrate, ไม่เลือกใหม่, ไม่โหลด bundle; คำนวณ metric ซ้ำจาก prediction รายแถวแล้วต้องตรงกับ 11 ภายใน tolerance มิฉะนั้นหยุด
- ช่วงข้อมูลแยกกันเสมอ: pooled OOF 2022–2023 (หลักฐานการเลือก), 2024 development (เคยเห็นแล้ว), 2025 reused (เคยเห็นแล้ว) — ไม่มี untouched test; daily และ 6h แยกกัน
- เอาต์พุต: `reports/v2/analysis/*` (aggregate CSV, PNG, `analysis_report_th.md`, `model_card_v2_analysis_th.md`, `analysis_manifest.json`) และ
  `data/v2/analysis/analysis_rows_<unit>.csv.gz` (รายแถว, gitignored)

Run All ใน VS Code ด้วย kernel `.venv` หรือ `.venv/bin/python backend/scripts/run_v2_notebooks.py --stage evaluate`

In [ ]:
# SETUP: bound numerical thread pools and select a non-interactive plotting backend BEFORE numpy / matplotlib are imported (no fitting happens in 12)
import os
for _k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '2'
os.environ.setdefault('MPLBACKEND', 'Agg')
print({k: os.environ[k] for k in ('OMP_NUM_THREADS', 'MPLBACKEND')})

In [ ]:
# FUNCTIONS (notebook 12, part 1: I/O + hashes, roots, notebook-11 completion / experiment lock / selection lock / output-hash verification, read-only snapshots, prediction + context loading) - self-contained; tests execute both FUNCTIONS cells in order (no network, no fit)
import datetime as dt, gzip, hashlib, json, math, os, sys
from pathlib import Path
import numpy as np, pandas as pd

MODEL_MANIFEST_SCHEMA = 'roadaware-v2-model-manifest/1'
MODEL_PRODUCER = '11_v2_model_experiments'
SELECTION_LOCK_SCHEMA = 'roadaware-v2-selection-lock/1'
ANALYSIS_SCHEMA = 'roadaware-v2-analysis-manifest/1'
ANALYSIS_PRODUCER = '12_v2_evaluation_error_analysis'
RUNNER_LOG_MODEL = 'run_manifest_model.json'   # runner log of a 11 run, rewritten after every cell (mutable, never a 11 result): canonical location
                                               # <reports/v2>/ (parent of the model report dir); older runners wrote it INTO reports/v2/models (legacy)
REQUIRED_REPORTS = ('oof_metrics_pooled.csv', 'oof_metrics_by_fold.csv', 'ablation_deltas.csv', 'fold_boundaries.csv', 'fit_log.csv',
                    'diagnostic_metrics.csv', 'final_fit_log.csv', 'reload_check.csv', 'selection_lock.json', 'feature_missingness_by_split.csv')


class DependencyError(RuntimeError):
    """An input is missing, malformed, incomplete or differs from the hash its producer recorded: stop, never guess."""


class LockError(RuntimeError):
    """The experiment lock or the selection lock is absent, malformed or does not match the actual bytes."""


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False) + '\n', 'utf-8')
    os.replace(tmp, path)
    return path


def read_json_strict(path):
    return json.loads(Path(path).read_text('utf-8'), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f'{path}: non-strict JSON {c}')))


def write_csv(path, df):
    """Plain CSV, LF line endings; any infinite numeric value is refused (finite aggregates only)."""
    path = Path(path)
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError(f'{path.name}: infinite values')
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator='\n')
    return path


def write_csv_gz(path, df):
    """Gzip CSV with a fixed header (mtime 0, no stored name): identical tables give identical bytes."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as fh, gzip.GzipFile(filename='', mode='wb', fileobj=fh, mtime=0) as gz:
        gz.write(df.to_csv(index=False, lineterminator='\n', float_format='%.17g').encode('utf-8'))
    os.replace(tmp, path)
    return path


def verify_file(path, expected_sha256, label):
    """sha256 of an input must equal the hash recorded by its producer; raises DependencyError otherwise."""
    p = Path(path)
    if not p.is_file():
        raise DependencyError(f'{label}: missing input {p}')
    got = sha256_file(p)
    if not expected_sha256 or got != expected_sha256:
        raise DependencyError(f'{label}: sha256 {got[:12]} != recorded {str(expected_sha256)[:12]} ({p}); input changed since its producer ran')
    return got


def _cell_sources(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    return [(i, c) for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code']


def _src(c):
    return ''.join(c['source']) if isinstance(c['source'], list) else c['source']


def functions_cells_source(nb_path):
    """Concatenated source of every '# FUNCTIONS' code cell in notebook order (same definition notebook 11 hashes)."""
    hits = [_src(c) for _, c in _cell_sources(nb_path) if _src(c).lstrip().startswith('# FUNCTIONS')]
    if not hits:
        raise ValueError(f'{nb_path}: no FUNCTIONS cell')
    return '\n\n'.join(hits)


def code_cells_sha256(nb_path):
    """sha256 of every code cell source (cell index:id -> hash); outputs of an executed notebook do not change it."""
    return {f"{i}:{c.get('id', '')}": sha256_bytes(_src(c).encode('utf-8')) for i, c in _cell_sources(nb_path)}


# ---------- roots ----------
def find_backend():
    """ROADAWARE_BACKEND_ROOT, else the first cwd ancestor that is (or contains) backend/ with config/model_experiment_v2.json + notebooks/."""
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        for cand in (d, d / 'backend'):
            if (cand / 'config' / 'model_experiment_v2.json').is_file() and (cand / 'notebooks').is_dir():
                return cand
    raise FileNotFoundError('backend root not found from the working directory; set ROADAWARE_BACKEND_ROOT')


def env_path(name, default):
    v = os.environ.get(name)
    return Path(v).expanduser().resolve() if v else Path(default).resolve()


def inside(p, root):
    p, root = Path(p).resolve(), Path(root).resolve()
    return p == root or root in p.parents


def check_write_roots(writes, read_only_dirs, read_only_files):
    """writes: [(label, dir)]; a write root may not equal/contain/sit inside a read-only dir and may not contain any read-only file.
    (data/v2/analysis may sit inside the 10 data root as long as it holds none of its files and is not the root itself.)"""
    for wl, w in writes:
        for rl, r in read_only_dirs:
            if inside(w, r) or inside(r, w):
                raise ValueError(f'{wl} {w} overlaps read-only {rl} {r}')
        hit = [str(f) for f in read_only_files if inside(f, w)]
        if hit:
            raise ValueError(f'{wl} {w} contains read-only inputs {hit[:3]}')
    return True


ANALYSIS_MARKERS = ('analysis_manifest.json',)


def existing_analysis(report_dir):
    return [str(Path(report_dir) / n) for n in ANALYSIS_MARKERS if (Path(report_dir) / n).exists()]


# ---------- notebook 11 completion, experiment lock, selection lock ----------
def verify_model_manifest(man, units):
    """The 11 manifest must say completed, research-only and carry the declared next_stage_interface for every unit."""
    if man.get('schema') != MODEL_MANIFEST_SCHEMA or man.get('producer') != MODEL_PRODUCER:
        raise DependencyError(f"model manifest schema/producer {man.get('schema')}/{man.get('producer')} unexpected")
    if man.get('status') != 'completed':
        raise DependencyError(f"notebook 11 status {man.get('status')!r} != 'completed': no analysis of an incomplete experiment")
    if man.get('production_ready') is not False or man.get('operational_status') != 'BLOCKED':
        raise DependencyError('model manifest must stay research-only (production_ready false, BLOCKED)')
    it = man.get('next_stage_interface') or {}
    need = {'oof_predictions', 'diagnostic_predictions', 'panels_and_features', 'methods', 'selected', 'selection_lock', 'deviance_floor',
            'diagnostic_split_labels'}
    if not need <= set(it):
        raise DependencyError(f'next_stage_interface lacks {sorted(need - set(it))}')
    for u in units:
        for part in ('oof_predictions', 'diagnostic_predictions', 'panels_and_features', 'methods', 'selected'):
            if u not in it[part]:
                raise DependencyError(f'next_stage_interface.{part} lacks unit {u}')
    if man.get('selection_lock', {}).get('sha256') != it['selection_lock'].get('sha256'):
        raise DependencyError('selection lock hash differs between the manifest body and next_stage_interface')
    return True


def model_runner_log_paths(model_rep):
    """Where a 11 runner log can be: canonical = model report parent (reports/v2), legacy = inside the model report dir."""
    rep = Path(model_rep)
    return {'canonical': rep.parent / RUNNER_LOG_MODEL, 'legacy_report_local': rep / RUNNER_LOG_MODEL}


def verify_runner_log(path, notebook=MODEL_PRODUCER, model_rep=None):
    """A 11 runner log must report the whole run and notebook 11 (including its final checks cell) as ok; missing/running/failed stops.
    When model_rep is given, the log must belong to that model report dir."""
    p = Path(path)
    if not p.is_file():
        raise DependencyError(f'runner log {p} not found: completion of every 11 cell (including its final checks) is unproven')
    log = read_json_strict(p)
    nbs = [e for e in log.get('notebooks', []) if e.get('notebook') == notebook]
    if log.get('status') != 'ok' or not nbs or nbs[-1].get('status') != 'ok':
        raise DependencyError(f"runner log {p.name}: run status {log.get('status')!r}, notebook {nbs[-1].get('status') if nbs else None!r}; "
                              'notebook 11 did not finish every cell (including its final checks)')
    if log.get('stage') != 'model':
        raise DependencyError(f"runner log {p}: stage {log.get('stage')!r} != 'model'")
    if model_rep is not None:
        logged = (log.get('roots') or {}).get('model_reports')
        if not logged or Path(logged).resolve() != Path(model_rep).resolve():
            raise DependencyError(f'runner log {p}: model report dir {logged!r} != {Path(model_rep).resolve()} (log of another 11 run)')
    return {'status': 'ok', 'path': str(p), 'started_utc': log.get('started_utc'), 'finished_utc': log.get('finished_utc'),
            'cells_completed': nbs[-1].get('cells_completed')}


def verify_model_runner_logs(model_rep, notebook=MODEL_PRODUCER):
    """11 completion per the runner: at least one log (canonical or legacy) must exist and EVERY available one must be ok for this dir."""
    found = {k: p for k, p in model_runner_log_paths(model_rep).items() if p.is_file()}
    if not found:
        raise DependencyError(f"no 11 runner log at {[str(p) for p in model_runner_log_paths(model_rep).values()]}: completion of every 11 cell "
                              '(including its final checks) is unproven; run 11 via backend/scripts/run_v2_notebooks.py --stage model')
    return {k: dict(verify_runner_log(p, notebook, model_rep), layout=k) for k, p in found.items()}


def verify_report_outputs(man, rep_dir, exclude=(RUNNER_LOG_MODEL,), required=REQUIRED_REPORTS):
    """Every 11 report file recorded in the manifest must exist with its recorded hash. Only the exact runner-owned log name is skipped
    (older 11 manifests listed the legacy report-local log); 11 may not declare any other file as excluded."""
    rec = {o['path']: o['sha256'] for o in man.get('report_outputs', [])}
    miss = [n for n in required if n not in rec]
    if miss:
        raise DependencyError(f'model manifest report_outputs lacks {miss}')
    bad = sorted(set(man.get('report_outputs_excluded') or {}) - set(exclude))
    if bad:
        raise DependencyError(f'model manifest excludes non-runner files from report_outputs: {bad}')
    out = {}
    for name, sha in sorted(rec.items()):
        if name in exclude:
            continue
        out[name] = verify_file(Path(rep_dir) / name, sha, f'11 report {name}')
    return out


def verify_model_data_outputs(man, model_data, units):
    rec = {(o['unit'], o['kind']): o for o in man.get('model_data_outputs', [])}
    for u in units:
        for kind in ('oof', 'diagnostic'):
            if (u, kind) not in rec:
                raise DependencyError(f'model manifest lacks {kind} predictions for {u}')
    return {f"{o['unit']}:{o['kind']}": verify_file(Path(model_data) / o['path'], o['sha256'], f"11 {o['kind']} predictions {o['unit']}")
            for o in rec.values()}


def verify_experiment_lock(lock_path, protocol_path, config_path, prep_manifest_path, prep_manifest, schema, fields):
    """Manager experiment lock: every recorded hash must equal the actual bytes (same contract as notebook 11)."""
    p = Path(lock_path)
    if not p.is_file():
        raise LockError(f'experiment lock {p} not found')
    lock = read_json_strict(p)
    missing = [k for k in fields if k not in lock]
    if missing:
        raise LockError(f'experiment lock lacks {missing}')
    if lock['schema'] != schema:
        raise LockError(f"experiment lock schema {lock['schema']!r} != {schema!r}")
    for key, path in (('protocol_sha256', protocol_path), ('model_config_sha256', config_path), ('preparation_manifest_sha256', prep_manifest_path)):
        got = sha256_file(path)
        if lock[key] != got:
            raise LockError(f'{key}: lock {str(lock[key])[:12]} != actual {got[:12]} ({path})')
    want = sorted((o['path'], o['sha256']) for o in prep_manifest['data_outputs'])
    have = sorted((o.get('path'), o.get('sha256')) for o in lock['preparation_data_outputs'])
    if have != want:
        raise LockError('lock preparation_data_outputs differ from the accepted 10 manifest data_outputs')
    return lock


def resolve_lock_path(entry_path, out11):
    p = Path(entry_path)
    return p if p.is_absolute() else Path(out11) / p


def verify_selection_lock(path, expected_sha, actual_hashes, out11):
    """Lock bytes, every file hash it records (relative to the 11 output root) and the given actual hashes must be unchanged."""
    p = Path(path)
    if not p.is_file():
        raise LockError(f'selection lock {p} missing')
    if sha256_file(p) != expected_sha:
        raise LockError('selection lock bytes differ from the hash recorded in the model manifest')
    lock = read_json_strict(p)
    if lock.get('schema') != SELECTION_LOCK_SCHEMA:
        raise LockError('selection lock schema mismatch')
    for e in lock.get('files', []):
        fp = resolve_lock_path(e['path'], out11)
        if not fp.is_file():
            raise LockError(f"{e['path']} recorded in the selection lock is missing ({fp})")
        got = sha256_file(fp)
        if got != e['sha256']:
            raise LockError(f"{e['path']} changed since the selection lock ({got[:12]} != {e['sha256'][:12]})")
    for k, v in actual_hashes.items():
        if lock.get('hashes', {}).get(k) != v:
            raise LockError(f'{k} differs from the selection lock')
    return lock


def compare_selection(lock_units, man_selection, iface_selected):
    """The locked choices, the manifest body and the 12 interface must agree exactly (12 never re-selects)."""
    for u, s in lock_units.items():
        m = man_selection.get(u, {})
        for k in ('recommended_method', 'recommendation_kind', 'best_candidate', 'strongest_baseline', 'family_best'):
            if s.get(k) != m.get(k) or s.get(k) != iface_selected.get(u, {}).get(k):
                raise LockError(f'{u}.{k} differs between selection lock, model manifest and next_stage_interface')
        if s['recommended_method'] != (s['best_candidate'] if s['gate']['passed'] else s['strongest_baseline']):
            raise LockError(f'{u}: recommendation does not follow the locked gate outcome')
    return True


# ---------- read-only snapshots ----------
def snapshot(paths):
    """{absolute path: sha256} of every read-only file (directories are expanded recursively)."""
    out = {}
    for p in paths:
        p = Path(p)
        files = sorted(x for x in p.rglob('*') if x.is_file()) if p.is_dir() else [p]
        for f in files:
            out[str(f)] = sha256_file(f)
    return out


def snapshot_diff(before, after):
    return sorted(k for k in set(before) | set(after) if before.get(k) != after.get(k))


# ---------- predictions + context ----------
def load_predictions(path, keys, methods, extra=()):
    """Saved 11 predictions with string keys; required columns present, keys unique, y a non-negative integer count, methods numeric."""
    df = pd.read_csv(path, dtype={k: str for k in keys}, keep_default_na=False, na_values=[''])
    need = list(dict.fromkeys(list(keys) + ['y'] + list(extra) + list(methods)))
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise DependencyError(f'{Path(path).name}: missing columns {miss}')
    if df.duplicated(list(keys)).any():
        raise DependencyError(f'{Path(path).name}: duplicate keys')
    y = df['y'].to_numpy(float)
    if np.isnan(y).any() or (y < 0).any() or not np.array_equal(y, np.floor(y)):
        raise DependencyError(f'{Path(path).name}: y must be a non-negative integer count')
    for m in methods:
        df[m] = pd.to_numeric(df[m], errors='raise').astype(float)
    return df


def load_context(panel_path, features_path, keys, feature_cols, target_col='y_recorded_count', mask_col='training_eligible', split_col='split'):
    """Accepted 10 panel (label, eligibility, split) and selected feature columns, row-aligned on identical keys (diagnostic use only)."""
    pan = pd.read_csv(panel_path, dtype=str, keep_default_na=False, usecols=lambda c: c in set(keys) | {target_col, mask_col, split_col})
    fe = pd.read_csv(features_path, dtype={k: str for k in keys}, keep_default_na=False, na_values=[''],
                     usecols=lambda c: c in set(keys) | set(feature_cols))
    miss = [c for c in list(keys) + list(feature_cols) if c not in fe.columns] + [c for c in (target_col, mask_col, split_col) if c not in pan.columns]
    if miss:
        raise DependencyError(f'context columns missing: {miss}')
    if len(fe) != len(pan) or not (fe[list(keys)].to_numpy() == pan[list(keys)].to_numpy()).all():
        raise DependencyError('feature keys differ from panel keys (row order must match exactly)')
    if pan.duplicated(list(keys)).any():
        raise DependencyError('duplicate panel keys')
    el = pan[mask_col].map({'True': True, 'False': False, 'true': True, 'false': False})
    if el.isna().any():
        raise DependencyError(f'{mask_col}: non-boolean values')
    ctx = fe[list(keys)].copy()
    ctx['ctx_eligible'] = el.astype(bool).to_numpy()
    ctx['ctx_split'] = pan[split_col].to_numpy()
    ctx['ctx_y'] = pd.to_numeric(pan[target_col], errors='raise').astype(float).to_numpy()
    for c in feature_cols:
        if c not in keys:
            ctx[c] = fe[c].to_numpy()
    return ctx


def join_context(pred, ctx, keys):
    """Many prediction rows -> one context row on identical keys; every prediction key must exist and be an eligible panel cell."""
    j = pred.merge(ctx, on=list(keys), how='left', validate='one_to_one', indicator=True)
    if not j['_merge'].eq('both').all():
        raise DependencyError(f"{int(j['_merge'].ne('both').sum())} prediction keys are absent from the accepted 10 panel")
    if not j['ctx_eligible'].all():
        raise DependencyError('a prediction row is not a training_eligible panel cell')
    if not np.array_equal(j['y'].to_numpy(float), j['ctx_y'].to_numpy(float)):
        raise DependencyError(f"{int((j['y'] != j['ctx_y']).sum())} saved labels differ from the accepted 10 panel label")
    return j.drop(columns='_merge')

In [ ]:
# FUNCTIONS (notebook 12, part 2: independent metrics, reconciliation with tolerance, selection/gate/bootstrap re-check, slice labels, weighted slices with support flags, tie-safe calibration bins, aggregate residuals, weekly series, paired ablations, coverage tables) - self-contained, no fitting
import math
import numpy as np, pandas as pd

METRIC_COLS = ['n', 'sum_y', 'sum_pred', 'mae', 'rmse', 'wape', 'bias', 'mean_poisson_deviance', 'n_deviance_floored']
WEEKDAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
FESTIVAL_WINDOWS = [(f'{fest}_{w}', f'fest_{fest}_{w}') for fest in ('songkran', 'new_year') for w in ('in', 'pre_1_3', 'pre_4_7', 'post_1_3', 'post_4_7')]
OUTCOME_BINS = [(0, 0, '0'), (1, 1, '1'), (2, 3, '2-3'), (4, 6, '4-6'), (7, 9, '7-9'), (10, math.inf, '10+')]


class ReconciliationError(RuntimeError):
    """An independently recomputed number differs from the notebook 11 aggregate beyond the declared tolerance."""


# ---------- metrics (written independently of notebook 11; same definitions as the locked protocol) ----------
def check_predictions(mu, label=''):
    mu = np.asarray(mu, dtype=float)
    if not np.isfinite(mu).all() or (mu < 0).any():
        raise ValueError(f'{label}: predictions must be finite and >= 0')
    return mu


def deviance_rows(y, mu, floor):
    """Unit Poisson deviance 2*(y*log(y/m) - (y - m)), m = max(mu, floor) ONLY here; y = 0 contributes 2*m. Returns (rows, n_floored)."""
    y, mu = np.asarray(y, dtype=float), check_predictions(mu, 'deviance')
    m = np.maximum(mu, floor)
    with np.errstate(divide='ignore', invalid='ignore'):
        t = np.where(y > 0, y * np.log(np.where(y > 0, y, 1.0) / m), 0.0)
    return 2.0 * (t - (y - m)), int((mu < floor).sum())


def metrics(y, mu, floor, deviance=True):
    """n, totals, MAE/RMSE/WAPE/bias on raw predictions; mean deviance (floored) unless deviance=False. Empty input -> n=0 and nulls."""
    y, mu = np.asarray(y, dtype=float), np.asarray(mu, dtype=float)
    out = {'n': int(len(y)), 'sum_y': float(y.sum()), 'sum_pred': float(mu.sum()), 'mae': None, 'rmse': None, 'wape': None, 'bias': None,
           'mean_poisson_deviance': None, 'n_deviance_floored': None}
    if len(y) == 0:
        return out
    mu = check_predictions(mu, 'metrics')
    e = mu - y
    out.update(mae=float(np.abs(e).mean()), rmse=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()),
               wape=None if y.sum() == 0 else float(np.abs(e).sum() / y.sum()))
    if deviance:
        d, nf = deviance_rows(y, mu, floor)
        out.update(mean_poisson_deviance=float(d.mean()), n_deviance_floored=nf)
    return out


def is_zero(mid):
    return mid == 'baseline_zero'


def method_meta(mid, cfg_family):
    """cfg_family: config_id -> family."""
    if mid.startswith('baseline_'):
        return {'family': 'baseline', 'config_id': mid[len('baseline_'):], 'variant': '(none)'}
    cid, v = mid.split('__', 1)
    return {'family': cfg_family[cid], 'config_id': cid, 'variant': v}


def recompute_tables(oof, methods, cfg_family, floor, failed=()):
    """Pooled + per-fold metrics of every method from the raw saved OOF rows; a method with any NaN (or in failed) gets status failed."""
    pooled, byfold = [], []
    y = oof['y'].to_numpy(float)
    for mid in methods:
        meta = method_meta(mid, cfg_family)
        mu = oof[mid].to_numpy(float)
        if mid in failed or np.isnan(mu).any():
            pooled.append(dict(method_id=mid, **meta, status='failed'))
            continue
        pooled.append(dict(method_id=mid, **meta, status='ok', **metrics(y, mu, floor, deviance=not is_zero(mid))))
        for fold, g in oof.groupby('fold', sort=False):
            byfold.append(dict(method_id=mid, **meta, fold=fold, **metrics(g['y'], g[mid], floor, deviance=not is_zero(mid))))
    return pd.DataFrame(pooled), pd.DataFrame(byfold)


def _isnull(v):
    return v is None or (isinstance(v, float) and math.isnan(v)) or (v is pd.NA)


def reconcile(mine, ref, keys, cols=METRIC_COLS, rtol=1e-9, atol=1e-9):
    """Long table key x metric with value_12, value_11, abs_diff and pass (|a-b| <= atol + rtol*|b|; both null -> pass; one null -> fail).
    Rows present on one side only are reported as failures (metric = '(row)')."""
    a = mine[list(keys) + [c for c in cols if c in mine.columns]].copy()
    b = ref[list(keys) + [c for c in cols if c in ref.columns]].copy()
    for df in (a, b):
        for k in keys:
            df[k] = df[k].astype(str)
    m = a.merge(b, on=list(keys), how='outer', suffixes=('_12', '_11'), indicator=True)
    rows = []
    for r in m.to_dict('records'):
        kv = {k: r[k] for k in keys}
        if r['_merge'] != 'both':
            rows.append(dict(kv, metric='(row)', value_12=None, value_11=None, abs_diff=None, passed=False, note=f"only in {r['_merge']}"))
            continue
        for c in cols:
            x, y = r.get(f'{c}_12'), r.get(f'{c}_11')
            if _isnull(x) and _isnull(y):
                ok, d = True, None
            elif _isnull(x) or _isnull(y):
                ok, d = False, None
            else:
                x, y = float(x), float(y)
                d = abs(x - y)
                ok = d <= atol + rtol * abs(y)
            rows.append(dict(kv, metric=c, value_12=x, value_11=y, abs_diff=d, passed=bool(ok), note=''))
    return pd.DataFrame(rows)


def assert_reconciled(table, label):
    bad = table[~table['passed']]
    if len(bad):
        raise ReconciliationError(f'{label}: {len(bad)} mismatches beyond tolerance, e.g. {bad.head(5).to_dict("records")}')
    return True


# ---------- selection / gate / bootstrap re-check (verification only; never changes the locked choice) ----------
def pick_best(rows, family_order=()):
    ok = [r for r in rows if r.get('status') == 'ok' and not _isnull(r.get('mean_poisson_deviance'))]
    if not ok:
        return None
    rank = {f: i for i, f in enumerate(family_order)}
    return min(ok, key=lambda r: (r['mean_poisson_deviance'], rank.get(r['family'], len(rank)), r['method_id']))


def reselect_for_check(pooled, family_order):
    rows = pooled.to_dict('records')
    fam = {f: pick_best([r for r in rows if r['family'] == f]) for f in family_order}
    cand = pick_best([r for r in fam.values() if r is not None], family_order)
    base = pick_best([r for r in rows if r['family'] == 'baseline' and not is_zero(r['method_id'])])
    return {'family_best': {f: (None if r is None else r['method_id']) for f, r in fam.items()},
            'best_candidate': None if cand is None else cand['method_id'], 'strongest_baseline': None if base is None else base['method_id']}


def calendar_blocks(dates, anchor, block_days):
    d = pd.to_datetime(pd.Series(dates).astype(str), format='%Y-%m-%d')
    return ((d - pd.Timestamp(anchor)).dt.days // int(block_days)).to_numpy()


def block_bootstrap(diff, blocks, n_resamples, seed, ci=0.95):
    """Paired calendar-block bootstrap of the row-weighted mean difference (whole blocks resampled together), same scheme as the protocol."""
    diff = np.asarray(diff, dtype=float)
    ub, inv = np.unique(np.asarray(blocks), return_inverse=True)
    sums, cnts = np.bincount(inv, weights=diff), np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(ub), size=(int(n_resamples), len(ub)))
    stats = sums[idx].sum(axis=1) / cnts[idx].sum(axis=1)
    a = (1 - ci) / 2
    return {'point': float(diff.mean()), 'lower': float(np.quantile(stats, a)), 'upper': float(np.quantile(stats, 1 - a)), 'n_blocks': int(len(ub)),
            'n_resamples': int(n_resamples), 'seed': int(seed), 'ci': ci}


def gate_checks(cand, base, cand_folds, base_folds, boot, g):
    rel = (base['mean_poisson_deviance'] - cand['mean_poisson_deviance']) / base['mean_poisson_deviance']
    wins = sum(cand_folds[f] < base_folds[f] for f in base_folds)
    checks = {'relative_deviance_reduction': {'value': rel, 'threshold': g['min_relative_deviance_reduction'], 'pass': bool(rel >= g['min_relative_deviance_reduction'])},
              'mae_ratio': {'value': cand['mae'] / base['mae'] if base['mae'] > 0 else None, 'threshold': g['max_mae_ratio'],
                            'pass': bool(cand['mae'] <= g['max_mae_ratio'] * base['mae'])},
              'fold_wins': {'value': int(wins), 'of': len(base_folds), 'threshold': g['min_fold_wins'], 'pass': bool(wins >= g['min_fold_wins'])},
              'bootstrap_upper_ci_lt_0': {'value': boot['upper'], 'threshold': 0.0, 'pass': bool(boot['upper'] < 0)}}
    return {'passed': all(c['pass'] for c in checks.values()), 'checks': checks}


def paired_columns(df, a, b, keys):
    """Two methods compared on identical keys: unique keys, both columns complete. Returns the two arrays."""
    if df.duplicated(list(keys)).any():
        raise ValueError('paired comparison: duplicate keys')
    xa, xb = df[a].to_numpy(float), df[b].to_numpy(float)
    if np.isnan(xa).any() or np.isnan(xb).any():
        raise ValueError(f'paired comparison {a} vs {b}: incomplete predictions (keys would differ)')
    return xa, xb


def paired_on_keys(left, right, keys, col):
    """Join two separately saved prediction frames on identical key sets (no row may be missing on either side)."""
    l, r = left[list(keys) + [col]], right[list(keys) + [col]]
    if set(map(tuple, l[list(keys)].to_numpy())) != set(map(tuple, r[list(keys)].to_numpy())):
        raise ValueError('paired comparison: key sets differ')
    return l.merge(r, on=list(keys), suffixes=('_a', '_b'), validate='one_to_one')


# ---------- slice labels (diagnostics only; never model inputs, never causal) ----------
def _flag(s):
    s = pd.Series(s)
    if s.dtype == bool:
        return s.astype(float)
    return pd.to_numeric(s.map(lambda v: {'True': 1.0, 'False': 0.0, True: 1.0, False: 0.0}.get(v, v)), errors='coerce')


def festival_window(frame):
    """First matching festival window in a fixed order (in > pre/post), 'none' when every flag is 0, 'unknown' when a flag is missing.
    Returns (labels, n_rows_with_overlapping_windows)."""
    lab = pd.Series('none', index=frame.index, dtype=object)
    hits = pd.Series(0, index=frame.index)
    unknown = pd.Series(False, index=frame.index)
    for name, col in reversed(FESTIVAL_WINDOWS):
        if col not in frame.columns:
            continue
        v = _flag(frame[col]).to_numpy()
        on = v > 0.5
        lab[on] = name
        hits += on
        unknown |= np.isnan(v)
    lab[unknown & (hits == 0)] = 'unknown'
    return lab.to_numpy(), int((hits > 1).sum())


def availability(frame, cols):
    """'available' when every column is non-null, 'partial' when some are, 'missing' when none (retrospective source coverage)."""
    cols = [c for c in cols if c in frame.columns]
    if not cols:
        return np.array(['not_in_unit'] * len(frame), dtype=object)
    k = frame[cols].notna().sum(axis=1).to_numpy()
    return np.where(k == len(cols), 'available', np.where(k > 0, 'partial', 'missing')).astype(object)


def bank_status(s):
    v = _flag(s).to_numpy()
    return np.where(np.isnan(v), 'missing', np.where(v > 0.5, 'known_asof', 'not_known_asof')).astype(object)


def outcome_bin(y):
    y = np.asarray(y, dtype=float)
    out = np.empty(len(y), dtype=object)
    for lo, hi, lab in OUTCOME_BINS:
        out[(y >= lo) & (y <= hi)] = lab
    return out


def high_count_threshold(y_oof, q=0.99):
    """OOF-only high-count threshold (>= 1); applied unchanged to later years."""
    return max(1.0, float(np.ceil(np.quantile(np.asarray(y_oof, dtype=float), q))))


def add_slice_columns(j, wx_cols, trf_cols, high_thr):
    td = pd.to_datetime(j['target_date'].astype(str), format='%Y-%m-%d')
    j = j.copy()
    j['sl_year'] = td.dt.year.astype(str).to_numpy()
    j['sl_year_month'] = j['target_date'].astype(str).str[:7].to_numpy()
    j['sl_weekday'] = np.array(WEEKDAYS, dtype=object)[td.dt.dayofweek.to_numpy()]
    j['sl_festival'], n_overlap = festival_window(j)
    j['sl_bank_calendar'] = bank_status(j['bank_calendar_known_asof']) if 'bank_calendar_known_asof' in j.columns else 'not_in_unit'
    j['sl_weather'] = availability(j, wx_cols)
    j['sl_traffic'] = availability(j, trf_cols)
    j['sl_outcome_bin'] = outcome_bin(j['y'])
    j['sl_high_count'] = np.where(j['y'].to_numpy(float) >= high_thr, f'high_ge_{int(high_thr)}', f'below_{int(high_thr)}')
    if 'time_block' in j.columns:
        j['sl_time_block'] = j['time_block'].astype(str).to_numpy()
    return j, n_overlap


# ---------- weighted slices with support flags ----------
def support_flag(n, sum_y, min_cells, min_events):
    if n == 0:
        return 'empty'
    return 'adequate_support' if (n >= min_cells and sum_y >= min_events) else 'small_support_descriptive_only'


def slice_metrics(df, dim, methods, floor, min_cells, min_events, levels=None):
    """Per level of `dim` and method: support, actual/predicted totals, residual total (pred - actual), signed bias, MAE, RMSE, deviance.
    Declared `levels` absent from df appear as explicit empty rows (n = 0, metrics null). Totals are plain sums, so the level totals
    always add up to the overall total."""
    rows = []
    present = list(pd.unique(df[dim].astype(str)))
    order = list(levels) + [l for l in present if l not in set(levels)] if levels is not None else sorted(present)
    groups = {str(k): g for k, g in df.groupby(df[dim].astype(str), sort=False)}
    for lev in order:
        g = groups.get(lev)
        for m in methods:
            if g is None:
                r = metrics([], [], floor)
                r.update(n_days=0, share_zero=None)
            else:
                r = metrics(g['y'], g[m], floor, deviance=not is_zero(m))
                r.update(n_days=int(g['target_date'].nunique()), share_zero=float((g['y'] == 0).mean()))
            rows.append(dict(dimension=dim.replace('sl_', ''), level=lev, method_id=m, **r, residual_total=r['sum_pred'] - r['sum_y'],
                             support=support_flag(r['n'], r['sum_y'], min_cells, min_events)))
    return pd.DataFrame(rows)


# ---------- calibration bins (tie-safe; edges from OOF predictions only) ----------
def calibration_edges(pred, n_bins):
    """Unique quantile edges of the OOF predictions: repeated values collapse duplicate edges (never a qcut crash); one value -> one bin."""
    p = np.asarray(pred, dtype=float)
    if len(p) == 0:
        raise ValueError('calibration edges need at least one prediction')
    return np.unique(np.quantile(p, np.linspace(0.0, 1.0, int(n_bins) + 1)))


def calibration_table(y, mu, edges, min_cells, min_events):
    """Bins [e0,e1), ..., [e_{k-1}, e_k]; values outside the OOF range fall in the first/last bin (open ends recorded). Empty bins kept."""
    y, mu = np.asarray(y, dtype=float), np.asarray(mu, dtype=float)
    edges = np.asarray(edges, dtype=float)
    nb = max(len(edges) - 1, 1)
    idx = np.searchsorted(edges[1:-1], mu, side='right') if len(edges) > 2 else np.zeros(len(mu), dtype=int)
    rows = []
    for b in range(nb):
        m = idx == b
        n, sy, sp = int(m.sum()), float(y[m].sum()), float(mu[m].sum())
        lo = float(edges[b]) if len(edges) > 1 else float(edges[0])
        hi = float(edges[b + 1]) if len(edges) > 1 else float(edges[0])
        rows.append({'bin': b, 'edge_lo': lo, 'edge_hi': hi, 'n': n, 'sum_pred': sp, 'sum_y': sy, 'mean_pred': sp / n if n else None,
                     'mean_y': sy / n if n else None, 'obs_over_pred': (sy / sp) if sp > 0 else None,
                     'pred_min': float(mu[m].min()) if n else None, 'pred_max': float(mu[m].max()) if n else None,
                     'support': support_flag(n, sy, min_cells, min_events)})
    return pd.DataFrame(rows)


# ---------- aggregate residuals, weekly series ----------
def aggregate_residuals(df, by, method, top, min_cells, min_events):
    g = df.groupby(list(by), sort=True).agg(n=('y', 'size'), sum_y=('y', 'sum'), sum_pred=(method, 'sum')).reset_index()
    g['residual_total'] = g['sum_pred'] - g['sum_y']
    g['abs_residual'] = g['residual_total'].abs()
    g['support'] = [support_flag(n, s, min_cells, min_events) for n, s in zip(g['n'], g['sum_y'])]
    g = g.sort_values(['abs_residual'] + list(by), ascending=[False] + [True] * len(by), kind='mergesort').head(int(top))
    return g.assign(method_id=method).reset_index(drop=True)


def weekly_totals(df, methods):
    """Sum of actual and predicted counts per Monday-starting week over the eligible cells of the frame (cells, not provinces, are counted)."""
    td = pd.to_datetime(df['target_date'].astype(str), format='%Y-%m-%d')
    wk = (td - pd.to_timedelta(td.dt.dayofweek, unit='D')).dt.strftime('%Y-%m-%d')
    agg = {'n_cells': ('y', 'size'), 'sum_y': ('y', 'sum')}
    agg.update({f'sum_pred__{m}': (m, 'sum') for m in methods})
    return df.assign(week_start=wk.to_numpy()).groupby('week_start', sort=True).agg(**agg).reset_index()


# ---------- paired ablations on identical keys ----------
def ablation_rows(oof, pairs, config_ids, cfg_family, floor, blocks, boot, keys, label='pooled_oof'):
    """For each configuration and (variant, reference): row-paired deviance difference on the identical OOF keys, MAE change, folds
    improved and an exploratory paired block-bootstrap interval. Incomplete methods are reported, not dropped silently."""
    out = []
    y = oof['y'].to_numpy(float)
    folds = list(pd.unique(oof['fold']))
    for cid in config_ids:
        for v_new, v_ref in pairs:
            a, b = f'{cid}__{v_new}', f'{cid}__{v_ref}'
            base = {'scope': label, 'config_id': cid, 'family': cfg_family[cid], 'variant': v_new, 'reference': v_ref}
            if a not in oof.columns or b not in oof.columns or oof[a].isna().any() or oof[b].isna().any():
                out.append(dict(base, status='incomplete'))
                continue
            xa, xb = paired_columns(oof, a, b, keys)
            da, _ = deviance_rows(y, xa, floor)
            db, _ = deviance_rows(y, xb, floor)
            diff = da - db
            fi, nf = 0, 0
            for f in folds:
                m = (oof['fold'] == f).to_numpy()
                if m.any():  # a coverage subset may have no row in some fold
                    nf += 1
                    fi += int(da[m].mean() < db[m].mean())
            bs = block_bootstrap(diff, blocks, boot['n_resamples'], boot['seed'], boot['ci'])
            delta = float(diff.mean())
            out.append(dict(base, status='ok', n_rows=int(len(y)), deviance_variant=float(da.mean()), deviance_reference=float(db.mean()),
                            delta_deviance=delta, relative_change=delta / float(db.mean()),
                            delta_mae=float(np.abs(xa - y).mean() - np.abs(xb - y).mean()), folds_improved=fi, n_folds=nf,
                            boot_lower=bs['lower'], boot_upper=bs['upper'],
                            effect='benefit' if delta < 0 else ('harm' if delta > 0 else 'no_change'),
                            interval='excludes_0' if (bs['upper'] < 0 or bs['lower'] > 0) else 'includes_0'))
    return pd.DataFrame(out)


def ablation_by_level(j, dim, pairs, config_ids, cfg_family, floor, anchor, boot, keys, min_cells, min_events):
    """Source-coverage sensitivity: the same paired comparison restricted to each level of `dim`; both comparators always share the
    identical keys of that level. Descriptive: coverage is retrospective and confounded with year, not evidence of a feature effect."""
    parts = []
    for lev, g in j.groupby(dim, sort=True):
        g = g.reset_index(drop=True)
        r = ablation_rows(g, pairs, config_ids, cfg_family, floor, calendar_blocks(g['target_date'], anchor, boot['block_days']), boot, keys,
                          label=f'{dim.replace("sl_", "")}={lev}')
        r['support'] = support_flag(len(g), float(g['y'].sum()), min_cells, min_events)
        parts.append(r)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# ---------- coverage / drift ----------
def coverage_by_month(ctx, wx_cols, trf_cols):
    """Per year-month: all panel cells, eligible (label-known) cells and, among eligible cells, the share with weather / traffic / known
    bank calendar values. Describes retrospective source and label coverage, nothing causal."""
    c = ctx.copy()
    c['ym'] = c['target_date'].astype(str).str[:7]
    c['wx_avail'] = availability(c, wx_cols) == 'available'
    c['trf_avail'] = availability(c, trf_cols) == 'available'
    c['bank_known'] = bank_status(c['bank_calendar_known_asof']) == 'known_asof' if 'bank_calendar_known_asof' in c.columns else False
    rows = []
    for (ym, sp), g in c.groupby(['ym', 'ctx_split'], sort=True):
        e = g[g['ctx_eligible']]
        rows.append({'year_month': ym, 'split': sp, 'n_cells': int(len(g)), 'n_eligible': int(len(e)), 'eligible_share': float(len(e) / len(g)),
                     'weather_available_share_eligible': float(e['wx_avail'].mean()) if len(e) else None,
                     'traffic_available_share_eligible': float(e['trf_avail'].mean()) if len(e) else None,
                     'bank_known_share_eligible': float(e['bank_known'].mean()) if len(e) else None})
    return pd.DataFrame(rows)

In [ ]:
# PARAMETERS (fixed before reading any prediction; descriptive thresholds only - they never change the locked choice)
MIN_CELLS = 200      # a slice / bin / subset with fewer eligible cells is labelled small_support_descriptive_only
MIN_EVENTS = 30      # ... or with fewer recorded events
N_CAL_BINS = 10      # requested calibration bins (OOF prediction quantiles; duplicate edges collapse)
TOP_RESIDUALS = 15   # largest aggregate residuals listed per period / method / grouping
HIGH_Q = 0.99        # high-count threshold = ceil(OOF quantile) of the observed count, applied unchanged to 2024/2025
FIG_DPI = 110

## A. ตรวจ notebook 11 ว่าเสร็จสมบูรณ์ + lock + hash ทั้งหมด ก่อนอ่าน prediction

manifest 11 (`completed`, research-only), runner log ของ 11 (ตำแหน่งหลัก `reports/v2/run_manifest_model.json` หรือแบบเก่าใน `reports/v2/models/`; ต้องมีอย่างน้อยหนึ่งไฟล์และทุกไฟล์ที่มีต้อง ok), hash ของรายงาน/prediction/โค้ด 11, experiment lock, selection lock (ไบต์ + ไฟล์ที่ล็อก + hash อินพุตจริง), panel/feature ของ 10; snapshot ไฟล์อ่านอย่างเดียวก่อนเริ่ม

In [ ]:
# A. Roots, notebook 11 completion, experiment lock, selection lock and every recorded hash - verified BEFORE any prediction is read
import scipy, matplotlib
T_RUN0 = dt.datetime.now(dt.timezone.utc)
BACKEND = find_backend()
REPO = BACKEND.parent
CFG_PATH = BACKEND / 'config' / 'model_experiment_v2.json'
CFG = read_json_strict(CFG_PATH)
IC = CFG['inputs']
UNITS = list(CFG['units'])
NB11 = env_path('ROADAWARE_V2_MODEL_NOTEBOOK', BACKEND / 'notebooks' / '11_v2_model_experiments.ipynb')
NB12 = BACKEND / 'notebooks' / '12_v2_evaluation_error_analysis.ipynb'
OUT = env_path('ROADAWARE_V2_OUTPUT_ROOT', BACKEND)                                                  # 12 writes only below OUT
MODEL_REP = env_path('ROADAWARE_V2_MODEL_REPORT_DIR', BACKEND / CFG['outputs']['report_subdir'])     # read-only 11 reports
MODEL_DATA = env_path('ROADAWARE_V2_MODEL_DATA_ROOT', BACKEND / CFG['outputs']['model_data_subdir'])  # read-only 11 row predictions
OUT11 = MODEL_REP.parents[2]                                                                          # 11 output root (selection-lock paths)
PREP_DIR = env_path('ROADAWARE_V2_PREPARATION_DIR', BACKEND / IC['preparation_report_subdir'])
PREP_DATA = env_path('ROADAWARE_V2_DATA_ROOT', BACKEND / IC['preparation_data_subdir'])
LOCK_PATH = env_path('ROADAWARE_V2_EXPERIMENT_LOCK', REPO / IC['experiment_lock'])
PROTOCOL_PATH = REPO / IC['protocol_draft']
FCFG_PATH = BACKEND / IC['feature_config']
REP = OUT / 'reports' / 'v2' / 'analysis'                                                             # tracked aggregates + PNG
ADATA = env_path('ROADAWARE_V2_ANALYSIS_DATA_ROOT', OUT / 'data' / 'v2' / 'analysis')                # gitignored row diagnostics
V1 = env_path('ROADAWARE_V1_DATA_ROOT', BACKEND)
V1_PROTECTED = [V1 / 'data' / r for r in ('raw', 'interim', 'processed', 'model')] + [V1 / 'reports' / r for r in ('data', 'models')]
CHECKS = []  # only checks that actually executed are appended


def passed(name, detail=''):
    CHECKS.append({'check': name, 'status': 'passed', 'detail': detail})
    print('PASS', name, detail)


# accepted 10 manifest + manager experiment lock
PM_PATH = PREP_DIR / IC['preparation_manifest']
if not PM_PATH.is_file():
    raise DependencyError(f'accepted notebook 10 manifest not found: {PM_PATH}')
PM = read_json_strict(PM_PATH)
if PM.get('producer') != IC['preparation_producer'] or PM.get('status') not in IC['accepted_preparation_status'] or PM.get('production_ready') is not False:
    raise DependencyError('notebook 10 manifest is not an accepted research-only preparation run')
LOCK = verify_experiment_lock(LOCK_PATH, PROTOCOL_PATH, CFG_PATH, PM_PATH, PM, IC['experiment_lock_schema'], IC['experiment_lock_fields'])
passed('experiment_lock', 'protocol/config/10-manifest/data-output hashes equal the manager lock')

# notebook 11 completion
MAN_PATH = MODEL_REP / 'model_manifest.json'
if not MAN_PATH.is_file():
    raise DependencyError(f'notebook 11 manifest not found: {MAN_PATH}; run and verify notebook 11 first (12 never invents results)')
MAN = read_json_strict(MAN_PATH)
verify_model_manifest(MAN, UNITS)
IT = MAN['next_stage_interface']
RUNNER11 = verify_model_runner_logs(MODEL_REP)
_logs11 = ', '.join(k + ': ' + v['path'] for k, v in sorted(RUNNER11.items()))
passed('notebook11_completed', f'manifest status completed; runner log ok ({_logs11})')
REPORT11 = verify_report_outputs(MAN, MODEL_REP)
passed('notebook11_report_hashes', f'{len(REPORT11)} report files equal model_manifest.report_outputs (only the runner-owned log {RUNNER_LOG_MODEL} is never hashed: mutable)')
DATA11 = verify_model_data_outputs(MAN, MODEL_DATA, UNITS)
for u in UNITS:
    for part, kind in (('oof_predictions', 'oof'), ('diagnostic_predictions', 'diagnostic')):
        rec = [o for o in MAN['model_data_outputs'] if o['unit'] == u and o['kind'] == kind][0]
        if IT[part][u]['sha256'] != rec['sha256'] or IT[part][u]['path'] != rec['path']:
            raise DependencyError(f'next_stage_interface.{part}.{u} differs from model_data_outputs')
passed('notebook11_prediction_hashes', f'{len(DATA11)} prediction files equal model_data_outputs and next_stage_interface')

# notebook 11 code identity (code cells only; saved outputs of an executed notebook do not change these hashes)
FN11_SHA = sha256_bytes(functions_cells_source(NB11).encode('utf-8'))
if FN11_SHA != MAN['functions_cells_sha256'] or code_cells_sha256(NB11) != MAN['code_cells_sha256']:
    raise DependencyError('notebook 11 code cells differ from the code that produced the manifest')
passed('notebook11_code_hashes', f'FUNCTIONS {FN11_SHA[:12]} and every code cell equal the manifest')

# selection lock: bytes, recorded file hashes, actual input hashes; choices identical in lock / manifest / interface
ACTUAL_LOCK_HASHES = {'experiment_lock_sha256': sha256_file(LOCK_PATH), 'protocol_sha256': sha256_file(PROTOCOL_PATH),
                      'model_config_sha256': sha256_file(CFG_PATH), 'preparation_manifest_sha256': sha256_file(PM_PATH),
                      'notebook11_functions_sha256': FN11_SHA}
SEL_LOCK_PATH = MODEL_REP / IT['selection_lock']['path']
SEL_LOCK = verify_selection_lock(SEL_LOCK_PATH, IT['selection_lock']['sha256'], ACTUAL_LOCK_HASHES, OUT11)
compare_selection(SEL_LOCK['units'], MAN['selection'], IT['selected'])
if not MAN['selection_lock'].get('verified_after_diagnostics'):
    raise LockError('notebook 11 did not verify its selection lock after the 2024/2025 diagnostics')
SEL = SEL_LOCK['units']
passed('selection_lock', f'{SEL_LOCK_PATH.name} sha {IT["selection_lock"]["sha256"][:12]}; {len(SEL_LOCK["files"])} locked files unchanged; '
       'choices equal in lock/manifest/interface; recommendation follows the locked gate')

# accepted 10 panels/features (context only) + every 11 input role
PM_SHA = {o['path']: o['sha256'] for o in PM['data_outputs']}
IF10 = PM['next_stage_interface']
CTX_FILES = {}
for u in UNITS:
    pf = IT['panels_and_features'][u]
    for part in ('panel', 'features'):
        name = pf[part]
        if pf[f'{part}_sha256'] != PM_SHA.get(name):
            raise DependencyError(f'{u} {part}: 11 interface hash differs from the accepted 10 manifest')
        CTX_FILES[(u, part)] = PREP_DATA / name
        verify_file(PREP_DATA / name, PM_SHA[name], f'10 {u} {part}')
ROLE_PATHS = {'experiment_lock': LOCK_PATH, 'protocol_draft': PROTOCOL_PATH, 'model_config': CFG_PATH, 'preparation_manifest': PM_PATH,
              'feature_config': FCFG_PATH, **{f'{u}_{p}': CTX_FILES[(u, p)] for u in UNITS for p in ('panel', 'features')}}
for i in MAN['inputs']:
    if i['role'] in ROLE_PATHS and sha256_file(ROLE_PATHS[i['role']]) != i['sha256']:
        raise DependencyError(f"{i['role']} differs from the bytes notebook 11 used")
passed('notebook11_inputs_unchanged', f"{sum(i['role'] in ROLE_PATHS for i in MAN['inputs'])} 11 input roles hash-equal; 10 panels/features match the 10 manifest")

# write roots: never inside / around a read-only input; earlier 12 results are preserved
READ_ONLY_FILES = ([LOCK_PATH, PROTOCOL_PATH, CFG_PATH, FCFG_PATH, PM_PATH, NB11, MAN_PATH] + [MODEL_REP / n for n in REPORT11]
                   + [MODEL_DATA / o['path'] for o in MAN['model_data_outputs']] + list(CTX_FILES.values()) + [Path(v['path']) for v in RUNNER11.values()])
check_write_roots([('analysis report dir', REP), ('analysis data root', ADATA)],
                  [('11 report dir', MODEL_REP), ('11 model data root', MODEL_DATA), ('10 preparation dir', PREP_DIR)]
                  + [('protected v1', v) for v in V1_PROTECTED], READ_ONLY_FILES)
if inside(PREP_DATA, ADATA):
    raise ValueError('analysis data root equals or contains the read-only 10 data root')
_old = existing_analysis(REP)
if _old:
    raise RuntimeError(f'{_old} already exists (earlier notebook 12 run, preserved). Use a fresh root, e.g.\n'
                       '  .venv/bin/python backend/scripts/run_v2_notebooks.py --stage evaluate --output-dir backend/rerun-analysis-<UTC-stamp>\n'
                       'or in VS Code set ROADAWARE_V2_OUTPUT_ROOT=<fresh dir> before Run All.')
SNAP_BEFORE = snapshot(READ_ONLY_FILES)
passed('read_only_snapshot_before', f'{len(SNAP_BEFORE)} read-only files hashed')
REP.mkdir(parents=True, exist_ok=True)
ADATA.mkdir(parents=True, exist_ok=True)
WRITTEN = []  # every file 12 writes (hashed into the manifest)


def out_csv(name, df, gz=False):
    p = write_csv_gz(ADATA / name, df) if gz else write_csv(REP / name, df)
    WRITTEN.append(p)
    return p


VERSIONS = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'matplotlib': matplotlib.__version__}
pd.set_option('display.width', 200)
print(f'11 reports={MODEL_REP} (read-only)\n11 data={MODEL_DATA} (read-only)\n10 data={PREP_DATA} (read-only)\nreports -> {REP}\nrow diagnostics -> {ADATA}\nversions={VERSIONS}')

## B. คำนวณ metric ซ้ำจาก prediction รายแถวและกระทบยอดกับ 11

pooled OOF, ราย fold และ diagnostic 2024/2025 ของทุกวิธี; ป้ายต้องตรงกับ panel 10; ไม่ตรงเกิน tolerance = หยุด (`metric_reconciliation.csv`)

In [ ]:
# B. Load the hash-verified predictions, join the accepted 10 context on identical keys, recompute every metric and reconcile with 11
FLOOR = float(IT['deviance_floor'])
if FLOOR != float(CFG['metrics']['deviance_floor']):
    raise DependencyError('deviance floor differs between the 11 interface and the locked config')
TOL = {'rtol': 1e-9, 'atol': 1e-9}   # pure summation-order noise; anything larger is a mismatch and stops the notebook
CFG_FAMILY = {c['id']: c['family'] for c in CFG['model_configs']}
FOLD_NAMES = [f['name'] for f in CFG['folds']]
DIAG_SPLITS = [s['name'] for s in CFG['final_fit']['diagnostic_splits']]
FITLOG11 = pd.read_csv(MODEL_REP / 'fit_log.csv')
POOLED11 = pd.read_csv(MODEL_REP / 'oof_metrics_pooled.csv')
BYFOLD11 = pd.read_csv(MODEL_REP / 'oof_metrics_by_fold.csv')
DMET11 = pd.read_csv(MODEL_REP / 'diagnostic_metrics.csv')
ABL11 = pd.read_csv(MODEL_REP / 'ablation_deltas.csv')
GROUPS10 = IF10['feature_groups']
WX_COLS = {u: [c for c in GROUPS10['weather'] if c in IF10[u]['feature_columns']] for u in UNITS}
TRF_COLS = {u: [c for c in GROUPS10['traffic'] if c in IF10[u]['feature_columns']] for u in UNITS}
FEST_COLS = [c for _, c in FESTIVAL_WINDOWS]
OOF, DIAG, CTX, POOLED12, BYFOLD12, DMET12, RECON = {}, {}, {}, {}, {}, {}, []
for u in UNITS:
    ito, itd = IT['oof_predictions'][u], IT['diagnostic_predictions'][u]
    keys = list(ito['keys'])
    oof = load_predictions(MODEL_DATA / ito['path'], keys, ito['method_columns'], extra=['fold', 'target_date'])
    dg = load_predictions(MODEL_DATA / itd['path'], keys, itd['method_columns'], extra=['split', 'target_date'])
    if len(oof) != ito['rows'] or len(dg) != itd['rows']:
        raise DependencyError(f'{u}: row counts differ from the interface')
    if list(pd.unique(oof['fold'])) != FOLD_NAMES or not set(dg['split']) <= set(DIAG_SPLITS):
        raise DependencyError(f'{u}: unexpected folds/splits')
    ctx = load_context(CTX_FILES[(u, 'panel')], CTX_FILES[(u, 'features')], keys,
                       ['province'] + FEST_COLS + ['bank_calendar_known_asof'] + WX_COLS[u] + TRF_COLS[u], target_col=IF10[u]['target'],
                       mask_col=IF10[u]['mask'], split_col=IF10[u]['split'])
    CTX[u] = ctx
    OOF[u] = join_context(oof, ctx, keys)
    DIAG[u] = join_context(dg, ctx, keys)
    if not OOF[u]['ctx_split'].eq(CFG['time']['train_split']).all() or not (DIAG[u]['ctx_split'] == DIAG[u]['split']).all():
        raise DependencyError(f'{u}: prediction split labels disagree with the accepted 10 panel')
    # pooled + per-fold OOF metrics from raw rows
    failed = set(FITLOG11.loc[FITLOG11['unit'].eq(u) & FITLOG11['status'].ne('ok'), 'method_id'])
    pooled, byfold = recompute_tables(oof, ito['method_columns'], CFG_FAMILY, FLOOR, failed)
    POOLED12[u], BYFOLD12[u] = pooled.assign(unit=u), byfold.assign(unit=u)
    p11 = POOLED11[POOLED11['unit'].eq(u)]
    if dict(zip(pooled['method_id'], pooled['status'])) != dict(zip(p11['method_id'], p11['status'])):
        raise ReconciliationError(f'{u}: method status (ok/failed) differs from 11')
    ok = pooled['status'].eq('ok')
    RECON.append(reconcile(pooled[ok], p11[p11['status'].eq('ok')], ['method_id'], rtol=TOL['rtol'], atol=TOL['atol']).assign(unit=u, table='oof_metrics_pooled'))
    RECON.append(reconcile(byfold, BYFOLD11[BYFOLD11['unit'].eq(u)], ['method_id', 'fold'], rtol=TOL['rtol'], atol=TOL['atol']).assign(unit=u, table='oof_metrics_by_fold'))
    # 2024 development / 2025 reused diagnostics from raw rows (never pooled with OOF)
    dm = [dict(split=sp, method_id=m, **metrics(g['y'], g[m], FLOOR, deviance=not is_zero(m))) for sp, g in dg.groupby('split', sort=False)
          for m in itd['method_columns']]
    DMET12[u] = pd.DataFrame(dm).assign(unit=u)
    RECON.append(reconcile(DMET12[u], DMET11[DMET11['unit'].eq(u)], ['split', 'method_id'], rtol=TOL['rtol'], atol=TOL['atol']).assign(unit=u, table='diagnostic_metrics'))
    print(f"{u}: OOF rows {len(oof)} ({len(ito['method_columns'])} methods), diagnostic rows {dg['split'].value_counts().to_dict()}, "
          f"panel cells {len(ctx)} eligible {int(ctx['ctx_eligible'].sum())}")
RECON = pd.concat(RECON, ignore_index=True)
out_csv('metric_reconciliation.csv', RECON)
assert_reconciled(RECON, 'pooled / per-fold OOF and 2024/2025 diagnostic metrics')
passed('metric_reconciliation', f"{len(RECON)} recomputed values equal 11 within rtol {TOL['rtol']:g} / atol {TOL['atol']:g}; labels equal the 10 panel")
print(RECON.groupby(['unit', 'table']).agg(values=('passed', 'size'), max_abs_diff=('abs_diff', 'max')).to_string())

## C. หลักฐานการตัดสินใจ

ตัวแทนทุกตระกูล + strongest baseline, gate ที่ล็อกไว้ (คำนวณซ้ำต้องได้ผลเดิม), paired 7-day block bootstrap (สำรวจ), ชนะราย fold และเหตุผลของข้อเสนอ — ไม่มีการเปลี่ยนตัวเลือก

In [ ]:
# C. Decision evidence per unit: every family winner + strongest baseline, the locked gate re-checked, paired exploratory bootstrap, per-fold wins, reasons
G = CFG['gate']
BOOT = dict(G['bootstrap'])
ANCHOR = CFG['folds'][0]['eval_start']
DECISION, FOLDCMP, GATEROWS, REASONS, DECISION_JSON = [], [], [], {}, {}
for u in UNITS:
    s, oof = SEL[u], OOF[u]
    keys = list(IT['oof_predictions'][u]['keys'])
    pooled = POOLED12[u]
    chk = reselect_for_check(pooled, CFG['family_order'])
    if any(chk[k] != s[k] for k in ('family_best', 'best_candidate', 'strongest_baseline')):
        raise LockError(f'{u}: the locked choice does not follow from the recomputed pooled OOF metrics: {chk} vs lock')
    pm = pooled.set_index('method_id')
    fd = {m: g.set_index('fold')['mean_poisson_deviance'].to_dict() for m, g in BYFOLD12[u].groupby('method_id')}
    base_id, cand_id = s['strongest_baseline'], s['best_candidate']
    blocks = calendar_blocks(oof['target_date'], ANCHOR, BOOT['block_days'])
    y = oof['y'].to_numpy(float)
    d_base, _ = deviance_rows(y, oof[base_id], FLOOR) if base_id else (None, None)
    boots = {}
    for fam, mid in s['family_best'].items():
        if mid is None or base_id is None:
            continue
        xa, xb = paired_columns(oof, mid, base_id, keys)
        boots[mid] = block_bootstrap(deviance_rows(y, xa, FLOOR)[0] - d_base, blocks, BOOT['n_resamples'], BOOT['seed'], BOOT['ci'])
    # the gate's own bootstrap (best candidate) must reproduce the locked interval; the gate must reproduce the locked outcome
    if cand_id and base_id:
        b_lock = s['bootstrap']
        for k in ('point', 'lower', 'upper'):
            if abs(boots[cand_id][k] - b_lock[k]) > TOL['atol'] + TOL['rtol'] * abs(b_lock[k]):
                raise ReconciliationError(f'{u}: bootstrap {k} {boots[cand_id][k]} != locked {b_lock[k]}')
        if boots[cand_id]['n_blocks'] != b_lock['n_blocks']:
            raise ReconciliationError(f'{u}: bootstrap block count differs')
        gate = gate_checks(pm.loc[cand_id].to_dict(), pm.loc[base_id].to_dict(), fd[cand_id], fd[base_id], boots[cand_id], G)
        if gate['passed'] != s['gate']['passed'] or any(gate['checks'][k]['pass'] != s['gate']['checks'][k]['pass'] for k in gate['checks']):
            raise ReconciliationError(f'{u}: recomputed gate differs from the locked gate')
        for k, v in gate['checks'].items():
            GATEROWS.append({'unit': u, 'candidate': cand_id, 'baseline': base_id, 'check': k, 'value': v['value'], 'threshold': v['threshold'],
                             'passed': v['pass'], 'locked_value': s['gate']['checks'][k].get('value')})
    else:
        gate = s['gate']
    # decision table: all family winners, every baseline (zero = sparsity diagnostic), roles
    roles = {}
    for fam, mid in s['family_best'].items():
        if mid:
            roles.setdefault(mid, []).append(f'family_best:{fam}')
    for m, tag in ((cand_id, 'best_candidate'), (base_id, 'strongest_baseline'), (s['recommended_method'], 'RECOMMENDED')):
        if m:
            roles.setdefault(m, []).append(tag)
    show = [m for m in s['family_best'].values() if m] + list(pooled.loc[pooled['family'].eq('baseline'), 'method_id'])
    for m in dict.fromkeys(show):
        r = pm.loc[m]
        row = {'unit': u, 'method_id': m, 'family': r['family'], 'variant': r['variant'], 'roles': ' '.join(roles.get(m, [])), 'status': r['status']}
        row.update({k: r.get(k) for k in ('n', 'sum_y', 'sum_pred', 'mean_poisson_deviance', 'mae', 'rmse', 'bias', 'wape', 'n_deviance_floored')})
        if base_id and r['status'] == 'ok' and not is_zero(m):
            b = pm.loc[base_id]
            row.update(rel_deviance_reduction_vs_baseline=(b['mean_poisson_deviance'] - r['mean_poisson_deviance']) / b['mean_poisson_deviance'],
                       mae_ratio_vs_baseline=r['mae'] / b['mae'] if b['mae'] > 0 else None,
                       fold_wins_vs_baseline=int(sum(fd[m][f] < fd[base_id][f] for f in fd[base_id])))
        if m in boots:
            row.update(boot_point=boots[m]['point'], boot_lower=boots[m]['lower'], boot_upper=boots[m]['upper'],
                       boot_role='gate (locked)' if m == cand_id else 'exploratory extra, not part of the gate')
        DECISION.append(row)
    for m in dict.fromkeys([x for x in s['family_best'].values() if x] + ([base_id] if base_id else [])):
        for f in FOLD_NAMES:
            r = BYFOLD12[u][(BYFOLD12[u]['method_id'] == m) & (BYFOLD12[u]['fold'] == f)].iloc[0]
            FOLDCMP.append({'unit': u, 'fold': f, 'method_id': m, 'roles': ' '.join(roles.get(m, [])), 'n': r['n'], 'sum_y': r['sum_y'],
                            'mean_poisson_deviance': r['mean_poisson_deviance'], 'mae': r['mae'], 'bias': r['bias'],
                            'beats_strongest_baseline': None if m == base_id else bool(fd[m][f] < fd[base_id][f])})
    # plain-language reasons for the recommendation (from the locked gate; nothing is re-decided here)
    rs = []
    if not cand_id or not base_id:
        rs.append('no complete candidate or baseline: the protocol recommendation is the locked fallback')
    elif gate['passed']:
        rs.append(f'{cand_id} passed all four gate checks against {base_id}; recommended per protocol (research reference only)')
    else:
        rs.append(f'{cand_id} (best of the three families) did NOT pass the gate against {base_id}; the protocol keeps the strongest baseline as reference')
        for k, v in gate['checks'].items():
            if not v['pass']:
                rs.append(f"failed {k}: value {v['value']} vs threshold {v['threshold']}")
    REASONS[u] = rs
    DECISION_JSON[u] = {'recommended_method': s['recommended_method'], 'recommendation_kind': s['recommendation_kind'], 'best_candidate': cand_id,
                        'strongest_baseline': base_id, 'family_best': s['family_best'], 'gate_passed': bool(gate['passed']),
                        'gate_bootstrap': None if cand_id not in boots else {k: boots[cand_id][k] for k in ('point', 'lower', 'upper', 'n_blocks')},
                        'failed_methods': s.get('failed_methods'), 'reasons': rs}
DECISION, FOLDCMP, GATEROWS = pd.DataFrame(DECISION), pd.DataFrame(FOLDCMP), pd.DataFrame(GATEROWS)
out_csv('decision_summary.csv', DECISION)
out_csv('fold_comparison.csv', FOLDCMP)
out_csv('gate_checks.csv', GATEROWS)
passed('selection_and_gate_recheck', 'locked family winners / best candidate / strongest baseline follow from recomputed OOF; gate outcome and bootstrap interval reproduced')
print(DECISION[['unit', 'method_id', 'roles', 'mean_poisson_deviance', 'mae', 'fold_wins_vs_baseline', 'boot_lower', 'boot_upper']].to_string(index=False))
for u in UNITS:
    print(u, REASONS[u])

## D. การเพิ่ม feature บน keys เดียวกัน

ทุก configuration × ทุกคู่ใน protocol + ทุก variant เทียบ core ของ configuration เดียวกัน; แสดงทั้ง benefit และ harm; ความไวต่อความครอบคลุมของ weather/traffic ใช้ keys เดียวกันสำหรับทั้งสองวิธี (บรรยาย ไม่ใช่ผลเชิงสาเหตุ)

In [ ]:
# D. Feature additions on identical OOF keys: every configuration, every protocol comparison + each variant against the same configuration's core.
#    Benefits AND harms are kept; nothing is filtered by the result. Coverage sensitivity uses the same fixed keys for both comparators.
PROTO_PAIRS = [tuple(p) for p in CFG['ablation_comparisons']]
EXTRA_PAIRS = [(v, 'core') for v in CFG['feature_variants'] if v != 'core' and (v, 'core') not in PROTO_PAIRS]
CONFIG_IDS = [c['id'] for c in CFG['model_configs']]
ABL, ABL_SENS, ABL_RECON = [], [], []
for u in UNITS:
    oof, keys = OOF[u], list(IT['oof_predictions'][u]['keys'])
    blocks = calendar_blocks(oof['target_date'], ANCHOR, BOOT['block_days'])
    a = ablation_rows(oof, PROTO_PAIRS + EXTRA_PAIRS, CONFIG_IDS, CFG_FAMILY, FLOOR, blocks, BOOT, keys)
    a['comparison_source'] = ['protocol' if (v, r) in PROTO_PAIRS else 'additional_vs_core' for v, r in zip(a['variant'], a['reference'])]
    ABL.append(a.assign(unit=u))
    ref = ABL11[ABL11['unit'].eq(u) & ABL11['status'].eq('ok')]
    mine = a[a['comparison_source'].eq('protocol') & a['status'].eq('ok')]
    if len(mine) != len(ref) or set(zip(ref['config_id'], ref['variant'], ref['reference'])) != set(zip(mine['config_id'], mine['variant'], mine['reference'])):
        raise ReconciliationError(f'{u}: complete protocol ablation pairs differ from 11')
    ABL_RECON.append(reconcile(mine, ref, ['config_id', 'variant', 'reference'],
                               cols=['n_rows', 'deviance_variant', 'deviance_reference', 'delta_deviance', 'relative_change', 'delta_mae', 'folds_improved'],
                               rtol=TOL['rtol'], atol=TOL['atol']).assign(unit=u, table='ablation_deltas'))
    eff = mine.merge(ref, on=['config_id', 'variant', 'reference'], suffixes=('_12', '_11'))
    if not (eff['effect_12'] == eff['effect_11']).all():
        raise ReconciliationError(f'{u}: ablation benefit/harm labels differ from 11')
    # coverage sensitivity (descriptive): weather pairs split by weather availability, traffic pairs by traffic availability
    j, _ = add_slice_columns(oof, WX_COLS[u], TRF_COLS[u], 1.0)
    for dim, pairs in (('sl_weather', [('calendar_weather', 'calendar'), ('full', 'calendar_traffic')]),
                       ('sl_traffic', [('calendar_traffic', 'calendar'), ('full', 'calendar_weather')])):
        ABL_SENS.append(ablation_by_level(j, dim, pairs, CONFIG_IDS, CFG_FAMILY, FLOOR, ANCHOR, BOOT, keys, MIN_CELLS, MIN_EVENTS).assign(unit=u))
ABL, ABL_SENS, ABL_RECON = pd.concat(ABL, ignore_index=True), pd.concat(ABL_SENS, ignore_index=True), pd.concat(ABL_RECON, ignore_index=True)
out_csv('ablation_all_pairs.csv', ABL)
out_csv('ablation_coverage_sensitivity.csv', ABL_SENS)
out_csv('ablation_reconciliation.csv', ABL_RECON)
assert_reconciled(ABL_RECON, 'ablation deltas')
passed('ablation_reconciliation', f'{len(ABL_RECON)} protocol ablation values equal 11; benefit/harm labels equal; '
       f'{int(ABL["status"].eq("ok").sum())} paired comparisons on identical keys ({len(EXTRA_PAIRS)} additional vs-core pairs per configuration)')
print(ABL.groupby(['unit', 'comparison_source', 'effect']).size().to_string())
print(ABL[ABL['status'].eq('ok')].sort_values(['unit', 'delta_deviance'])[['unit', 'config_id', 'variant', 'reference', 'delta_deviance', 'relative_change',
                                                                         'folds_improved', 'boot_lower', 'boot_upper', 'effect']].to_string(index=False))

## E. Error slice, calibration, residual, อนุกรมเวลา และ drift

แต่ละช่วงแยกกัน; กลุ่มเล็ก/ว่างถูกระบุ; slice ตามผลลัพธ์ (0 / high count) และสถานะ missing เป็นการวินิจฉัยย้อนหลังเท่านั้น

In [ ]:
# E. Error slices, calibration bins, largest aggregate residuals, weekly actual-vs-predicted, label/source coverage and later-year drift.
#    Periods stay separate: pooled OOF (selection evidence), 2024 development (seen), 2025 reused (seen). Daily and 6 h stay separate.
SLICES, CALIB, RESID, WEEKLY, COVER, DRIFT, FOCUS, PERIOD_FRAMES, HIGH_THR, FEST_OVERLAP = [], [], [], [], [], [], {}, {}, {}, {}
PERIOD_LABEL = {'oof': 'pooled OOF 2022-2023 (selection evidence)', **IT['diagnostic_split_labels']}
for u in UNITS:
    s = SEL[u]
    FOCUS[u] = [m for m in dict.fromkeys([s['recommended_method'], s['best_candidate'], s['strongest_baseline']] + list(s['family_best'].values())) if m]
    HIGH_THR[u] = high_count_threshold(OOF[u]['y'], HIGH_Q)
    frames = {'oof': OOF[u]}
    frames.update({sp: DIAG[u][DIAG[u]['split'].eq(sp)].reset_index(drop=True) for sp in DIAG_SPLITS})
    ctx = CTX[u]
    prov_levels = sorted(ctx['province_key'].unique())
    PERIOD_FRAMES[u] = {}
    for per, fr in frames.items():
        j, FEST_OVERLAP[(u, per)] = add_slice_columns(fr, WX_COLS[u], TRF_COLS[u], HIGH_THR[u])
        PERIOD_FRAMES[u][per] = j
        years = sorted(ctx.loc[ctx['ctx_split'].eq('train' if per == 'oof' else per), 'target_date'].str[:4].unique())
        if per == 'oof':
            years = sorted({f['eval_start'][:4] for f in CFG['folds']})
        levels = {'sl_year': years, 'sl_year_month': [f'{y}-{m:02d}' for y in years for m in range(1, 13)], 'province_key': prov_levels,
                  'sl_weekday': WEEKDAYS, 'sl_festival': [n for n, _ in FESTIVAL_WINDOWS] + ['none', 'unknown'],
                  'sl_bank_calendar': ['known_asof', 'not_known_asof', 'missing'], 'sl_weather': ['available', 'partial', 'missing'],
                  'sl_traffic': ['available', 'partial', 'missing'], 'sl_outcome_bin': [lab for _, _, lab in OUTCOME_BINS],
                  'sl_high_count': [f'high_ge_{int(HIGH_THR[u])}', f'below_{int(HIGH_THR[u])}']}
        if 'sl_time_block' in j.columns:
            levels['sl_time_block'] = sorted(ctx['time_block'].astype(str).unique())
        tot = {m: (float(j['y'].sum()), float(j[m].sum()), len(j)) for m in FOCUS[u]}
        for dim, lev in levels.items():
            t = slice_metrics(j, dim, FOCUS[u], FLOOR, MIN_CELLS, MIN_EVENTS, levels=lev).assign(unit=u, period=per)
            # weighted slice totals must add up exactly to the period totals (every eligible row lands in exactly one level)
            for m, (sy, sp, n) in tot.items():
                g = t[t['method_id'].eq(m)]
                if int(g['n'].sum()) != n or abs(g['sum_y'].sum() - sy) > 1e-6 * max(1, sy) or abs(g['sum_pred'].sum() - sp) > 1e-6 * max(1, sp):
                    raise ReconciliationError(f'{u}/{per}/{dim}/{m}: slice totals do not add up to the period total')
            SLICES.append(t)
        for m in FOCUS[u]:
            if is_zero(m):
                continue
            edges = calibration_edges(OOF[u][m], N_CAL_BINS)   # edges from OOF only, applied unchanged to later years
            CALIB.append(calibration_table(j['y'], j[m], edges, MIN_CELLS, MIN_EVENTS).assign(unit=u, period=per, method_id=m, n_edges=len(edges)))
        for m in [x for x in dict.fromkeys([s['recommended_method'], s['best_candidate'], s['strongest_baseline']]) if x]:
            for by in (['province_key', 'sl_year_month'], ['target_date'], ['province_key']):
                RESID.append(aggregate_residuals(j, by, m, TOP_RESIDUALS, MIN_CELLS, MIN_EVENTS)
                             .assign(unit=u, period=per, grouping='+'.join(by)).rename(columns={'sl_year_month': 'year_month'}))
        WEEKLY.append(weekly_totals(j, FOCUS[u]).assign(unit=u, period=per))
        r0 = j[s['recommended_method']] if s['recommended_method'] else None
        DRIFT.append({'unit': u, 'period': per, 'period_label': PERIOD_LABEL[per], 'n_eligible_cells': len(j), 'n_days': int(j['target_date'].nunique()),
                      'first_date': j['target_date'].min(), 'last_date': j['target_date'].max(), 'mean_y': float(j['y'].mean()),
                      'share_zero': float((j['y'] == 0).mean()), f'share_y_ge_high': float((j['y'] >= HIGH_THR[u]).mean()),
                      'weather_available_share': float((j['sl_weather'] == 'available').mean()),
                      'traffic_available_share': float((j['sl_traffic'] == 'available').mean()),
                      'bank_known_share': float((j['sl_bank_calendar'] == 'known_asof').mean()),
                      'recommended_method': s['recommended_method'],
                      'recommended_bias': None if r0 is None else float((r0 - j['y']).mean()),
                      'recommended_sum_pred_over_sum_y': None if r0 is None or j['y'].sum() == 0 else float(r0.sum() / j['y'].sum()),
                      'festival_window_overlap_rows': FEST_OVERLAP[(u, per)]})
    cov = coverage_by_month(ctx, WX_COLS[u], TRF_COLS[u]).assign(unit=u)
    COVER.append(cov)
    # row diagnostics (gitignored): keys, period, slice labels, label and the focus predictions
    rows = pd.concat([PERIOD_FRAMES[u][p].assign(period=p) for p in PERIOD_FRAMES[u]], ignore_index=True)
    keep = list(IT['oof_predictions'][u]['keys']) + ['period', 'y'] + [c for c in rows.columns if c.startswith('sl_')] + FOCUS[u]
    out_csv(f'analysis_rows_{u}.csv.gz', rows[list(dict.fromkeys(keep))], gz=True)
SLICES = pd.concat(SLICES, ignore_index=True)
CALIB = pd.concat(CALIB, ignore_index=True)
RESID = pd.concat(RESID, ignore_index=True)
WEEKLY = pd.concat(WEEKLY, ignore_index=True)
COVER = pd.concat(COVER, ignore_index=True)
DRIFT = pd.DataFrame(DRIFT)
for u in UNITS:
    out_csv(f'slices_{u}.csv', SLICES[SLICES['unit'].eq(u)])
    out_csv(f'calibration_bins_{u}.csv', CALIB[CALIB['unit'].eq(u)])
    out_csv(f'largest_residuals_{u}.csv', RESID[RESID['unit'].eq(u)])
    out_csv(f'weekly_actual_vs_predicted_{u}.csv', WEEKLY[WEEKLY['unit'].eq(u)])
    out_csv(f'coverage_by_month_{u}.csv', COVER[COVER['unit'].eq(u)])
out_csv('drift_summary.csv', DRIFT)
passed('slice_totals', f'{len(SLICES)} slice rows; every dimension\'s level totals add up to its period total (n, actual, predicted)')
passed('calibration_bins', f'{len(CALIB)} bins from OOF-only unique quantile edges (ties collapse; empty bins kept with n = 0)')
print(DRIFT.to_string(index=False))
print(SLICES['support'].value_counts().to_dict())

## F. กราฟ PNG

In [ ]:
# F. Modest PNG figures (English axis text so every glyph renders without a Thai font; Thai explanation is in the report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 10, 'figure.dpi': FIG_DPI})
FIGS = []
PER_COLORS = {'oof': '#1f77b4', 'development_2024': '#ff7f0e', 'reused_seen_2025': '#2ca02c'}


def save_fig(fig, name):
    p = REP / name
    fig.tight_layout()
    fig.savefig(p, dpi=FIG_DPI)
    plt.close(fig)
    WRITTEN.append(p)
    FIGS.append(name)
    return p


for u in UNITS:
    s = SEL[u]
    rec, cand, base = s['recommended_method'], s['best_candidate'], s['strongest_baseline']
    # 1. pooled OOF deviance: every family winner + every nonzero baseline
    d = DECISION[DECISION['unit'].eq(u) & DECISION['mean_poisson_deviance'].notna()].sort_values('mean_poisson_deviance', ascending=False)
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(d) + 1.2))
    cols = ['#d62728' if m == rec else ('#9467bd' if m == cand else '#7f7f7f') for m in d['method_id']]
    ax.barh(d['method_id'], d['mean_poisson_deviance'], color=cols)
    lo = d['mean_poisson_deviance'].min()
    ax.set_xlim(lo * 0.97, d['mean_poisson_deviance'].max() * 1.01)
    ax.set_xlabel('pooled OOF mean Poisson deviance (lower is better; axis does not start at 0)')
    ax.set_title(f'{u}: family winners and baselines (red = recommended, purple = best candidate)')
    save_fig(fig, f'fig_oof_methods_{u}.png')
    # 2. per-fold deviance of family winners vs strongest baseline
    fc = FOLDCMP[FOLDCMP['unit'].eq(u)]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for m, g in fc.groupby('method_id', sort=False):
        ax.plot(g['fold'], g['mean_poisson_deviance'], marker='o', lw=2.2 if m in (base, rec) else 1.2, ls='--' if m == base else '-', label=m)
    ax.set_ylabel('mean Poisson deviance')
    ax.set_title(f'{u}: per-fold OOF deviance (dashed = strongest baseline)')
    ax.legend(fontsize=7)
    save_fig(fig, f'fig_fold_deviance_{u}.png')
    # 3. ablation: every configuration x comparison, benefit and harm, exploratory paired block interval
    a = ABL[ABL['unit'].eq(u) & ABL['status'].eq('ok')].copy()
    a['label'] = a['config_id'] + ': ' + a['variant'] + ' vs ' + a['reference']
    a = a.sort_values('relative_change')
    fig, ax = plt.subplots(figsize=(7.5, 0.22 * len(a) + 1.4))
    ref_dev = a['deviance_reference'].to_numpy()
    rel_lo, rel_hi = a['boot_lower'].to_numpy() / ref_dev, a['boot_upper'].to_numpy() / ref_dev
    yy = np.arange(len(a))
    ax.barh(yy, a['relative_change'] * 100, color=np.where(a['relative_change'] < 0, '#1f77b4', '#d62728'))
    rc = a['relative_change'].to_numpy()
    ax.errorbar(rc * 100, yy, xerr=[np.clip(rc - rel_lo, 0, None) * 100, np.clip(rel_hi - rc, 0, None) * 100], fmt='none', ecolor='k', lw=0.7)
    ax.set_yticks(yy)
    ax.set_yticklabels(a['label'], fontsize=6.5)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('relative change in pooled OOF deviance, % (blue = lower/better, red = higher/worse; bars = exploratory 95% block interval)')
    ax.set_title(f'{u}: feature additions on identical keys (all pairs, no filtering)')
    save_fig(fig, f'fig_ablation_{u}.png')
    # 4. calibration bins: observed vs predicted mean per bin, each period separately
    c = CALIB[CALIB['unit'].eq(u) & CALIB['n'].gt(0)]
    ms = [m for m in dict.fromkeys([rec, cand, base]) if m]
    fig, axes = plt.subplots(1, len(ms), figsize=(3.3 * len(ms), 3.3), squeeze=False)
    for ax, m in zip(axes[0], ms):
        g = c[c['method_id'].eq(m)]
        hi = max(g['mean_pred'].max(), g['mean_y'].max()) * 1.05
        ax.plot([0, hi], [0, hi], color='#999', lw=0.8)
        for per, gp in g.groupby('period', sort=False):
            small = gp['support'].ne('adequate_support')
            ax.plot(gp['mean_pred'], gp['mean_y'], marker='o', lw=1, color=PER_COLORS.get(per), label=per)
            ax.scatter(gp.loc[small, 'mean_pred'], gp.loc[small, 'mean_y'], facecolors='white', edgecolors=PER_COLORS.get(per), zorder=3)
        ax.set_title(m, fontsize=8)
        ax.set_xlabel('mean predicted per bin')
        ax.set_ylabel('mean observed per bin')
        ax.legend(fontsize=6)
    fig.suptitle(f'{u}: calibration bins (OOF edges; hollow = small support)', fontsize=9)
    save_fig(fig, f'fig_calibration_{u}.png')
    # 5. weekly actual vs predicted totals over eligible cells
    w = WEEKLY[WEEKLY['unit'].eq(u)].copy()
    w['week'] = pd.to_datetime(w['week_start'])
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for per, gp in w.groupby('period', sort=False):
        ax.plot(gp['week'], gp['sum_y'], color='k', lw=1.1, label='actual' if per == 'oof' else None)
        ax.plot(gp['week'], gp[f'sum_pred__{rec}'], color='#d62728', lw=1, label=f'recommended {rec}' if per == 'oof' else None)
        if cand and cand != rec:
            ax.plot(gp['week'], gp[f'sum_pred__{cand}'], color='#9467bd', lw=1, ls='--', label=f'best candidate {cand}' if per == 'oof' else None)
    ax.set_ylabel('weekly recorded count (eligible cells only)')
    ax.set_title(f'{u}: weekly actual vs predicted (OOF 2022-23 | 2024 development seen | 2025 reused seen; lines may bridge weeks without eligible cells)', fontsize=8)
    ax.legend(fontsize=7)
    save_fig(fig, f'fig_weekly_{u}.png')
    # 6. signed bias by year-month for the recommended method and best candidate
    sl = SLICES[SLICES['unit'].eq(u) & SLICES['dimension'].eq('year_month') & SLICES['method_id'].isin(ms) & SLICES['n'].gt(0)]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    for m, g in sl.groupby('method_id', sort=False):
        ax.plot(g['level'], g['bias'], marker='.', lw=1, label=m)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(range(0, sl['level'].nunique(), 3))
    ax.set_xticklabels(sorted(sl['level'].unique())[::3], rotation=60, fontsize=7)
    ax.set_ylabel('mean (predicted - actual) per cell')
    ax.set_title(f'{u}: signed bias by month (positive = over-prediction; months without eligible cells omitted)')
    ax.legend(fontsize=7)
    save_fig(fig, f'fig_bias_by_month_{u}.png')
    # 7. label and source coverage by month (retrospective coverage, not a feature effect)
    cv = COVER[COVER['unit'].eq(u)].sort_values('year_month')
    fig, ax = plt.subplots(figsize=(9, 3.2))
    xx = np.arange(len(cv))
    for col, lab in (('eligible_share', 'label-known (eligible) share of panel cells'), ('weather_available_share_eligible', 'weather available | eligible'),
                     ('traffic_available_share_eligible', 'traffic available | eligible'), ('bank_known_share_eligible', 'bank calendar known as-of | eligible')):
        ax.plot(xx, cv[col].astype(float), lw=1.2, label=lab)
    ax.set_xticks(xx[::3])
    ax.set_xticklabels(cv['year_month'].iloc[::3], rotation=60, fontsize=7)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f'{u}: retrospective label and source coverage by month')
    ax.legend(fontsize=7)
    save_fig(fig, f'fig_coverage_{u}.png')
passed('figures_written', f'{len(FIGS)} PNG files (visual inspection is a separate manual check, not done by the notebook)')
print(FIGS)

## G. รายงานไทย, model card (มุมมองหลังวิเคราะห์) และ manifest

In [ ]:
# G. Thai analysis report, updated V2 model card (analysis view) and the analysis manifest - every number below comes from the tables written above
def _f(x, nd=4):
    if x is None or (isinstance(x, float) and not math.isfinite(x)):
        return 'null'
    if isinstance(x, (bool, np.bool_)):
        return str(bool(x))
    if isinstance(x, (int, np.integer)):
        return f'{int(x):,}'
    if isinstance(x, (float, np.floating)):
        return f'{int(x):,}' if float(x).is_integer() and abs(x) >= 1 else f'{float(x):.{nd}f}'
    return str(x)


def _sup(v):
    return {'adequate_support': 'พอใช้', 'small_support_descriptive_only': '**ข้อมูลน้อย ห้ามตีความเป็นผล**', 'empty': 'ว่าง (ไม่มีเซลล์)'}.get(v, v)


def md_table(df, cols, heads):
    out = ['| ' + ' | '.join(heads) + ' |', '|' + '---|' * len(heads)]
    for r in df.reindex(columns=cols).to_dict('records'):
        out.append('| ' + ' | '.join(_sup(r[c]) if c == 'support' else _f(r[c]) for c in cols) + ' |')
    return out


# coverage facts computed from the accepted 10 panel (not copied from elsewhere)
COV_FACTS = {}
for u in UNITS:
    c = CTX[u]
    by = c.groupby('ctx_split').agg(cells=('ctx_eligible', 'size'), eligible=('ctx_eligible', 'sum')).to_dict('index')
    py = c.assign(year=c['target_date'].str[:4], wx=availability(c, WX_COLS[u]) != 'missing').groupby(['province_key', 'year'])['wx'].any()
    COV_FACTS[u] = {'cells_by_split': {k: {'cells': int(v['cells']), 'eligible': int(v['eligible'])} for k, v in by.items()},
                    'province_years_with_any_weather': int(py.sum()), 'province_years_total': int(len(py)),
                    'weather_province_years_by_year': {y: int(g.sum()) for y, g in py.groupby(level='year')}}

L = ['# รายงานวิเคราะห์ผลการทดลอง V2 และ error analysis (Notebook 12)', '',
     '**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED.** ไม่ใช่ Dashboard/เว็บที่ deploy, ไม่ใช่ความเสี่ยงรายบุคคลหรือรายเส้นทาง, '
     'ไม่ใช่อุบัติเหตุทุกถนน (ป้าย = จำนวนบันทึก MOT ที่ถูกเลือก) และ **ไม่มี untouched test**', '',
     'ช่วงข้อมูล 3 แบบแยกกันเสมอ ไม่รวมกัน: (1) **pooled OOF 2022–2023** = หลักฐานที่ใช้เลือกโมเดล; (2) **2024 development** = เคยเห็นแล้วระหว่างพัฒนา; '
     '(3) **2025 reused** = เคยใช้/เคยเห็นแล้ว — (2)(3) เป็น diagnostic ไม่ได้ใช้เปลี่ยนตัวเลือก, daily และ 6h วิเคราะห์แยกกัน ไม่เทียบ error ข้ามหน่วย', '',
     '## 1. การตรวจสอบที่รันจริงก่อน/ระหว่างวิเคราะห์', '']
L += [f"- {c['check']}: {c['detail']}" for c in CHECKS]
L += ['', f"ไม่มีการ fit, tune, calibrate หรือเลือกใหม่; ค่าที่คำนวณซ้ำทั้งหมดต้องตรงกับ notebook 11 ภายใน rtol {TOL['rtol']:g} / atol {TOL['atol']:g} "
      '(ไม่ตรง = หยุดทำงาน) ดู `metric_reconciliation.csv`, `ablation_reconciliation.csv`', '']
for k, u in enumerate(UNITS, 1):
    s, dj = SEL[u], DECISION_JSON[u]
    L += [f'## 2.{k} หน่วย {u}: ตัวเลือกและเหตุผล', '',
          f"- **ข้อเสนอตาม protocol: `{dj['recommended_method']}`** ({dj['recommendation_kind']})",
          f"- best candidate จาก 3 ตระกูล: `{dj['best_candidate']}`; strongest baseline: `{dj['strongest_baseline']}`; ตัวแทนแต่ละตระกูล: {dj['family_best']}",
          f"- วิธีที่ล้มเหลว/ไม่ครบ fold: {dj['failed_methods'] or 'ไม่มี'}", '']
    d = DECISION[DECISION['unit'].eq(u)]
    L += md_table(d, ['method_id', 'roles', 'mean_poisson_deviance', 'mae', 'rmse', 'bias', 'rel_deviance_reduction_vs_baseline', 'fold_wins_vs_baseline',
                      'boot_lower', 'boot_upper'],
                  ['วิธี', 'บทบาท', 'deviance OOF', 'MAE', 'RMSE', 'bias', 'ลด deviance vs baseline', 'ชนะ fold', 'boot ล่าง', 'boot บน'])
    L += ['', 'boot = ช่วง 95% ของผลต่าง deviance (วิธี − strongest baseline) แบบ paired 7-day block bootstrap; ใช้ใน gate เฉพาะ best candidate, '
          'ของตระกูลอื่นเป็นข้อมูลสำรวจเพิ่ม ไม่ปรับ multiple comparison', '', '**Gate (ค่าที่ล็อกไว้และคำนวณซ้ำได้ตรงกัน):**', '']
    g = GATEROWS[GATEROWS['unit'].eq(u)] if len(GATEROWS) else GATEROWS
    if len(g):
        L += md_table(g, ['check', 'value', 'threshold', 'passed'], ['เกณฑ์', 'ค่า', 'เกณฑ์ผ่าน', 'ผ่าน'])
    L += [''] + [f'- {r}' for r in REASONS[u]]
    if not dj['gate_passed']:
        L += ['', f"**สรุปตรง ๆ: ไม่มีโมเดลจาก 3 ตระกูลที่ผ่านเกณฑ์ชนะ baseline อย่างมั่นคงในหน่วย {u}; จึงคง strongest baseline `{dj['strongest_baseline']}` "
              'เป็น reference ตาม protocol — ไม่ได้ปรับแต่งหรือเปลี่ยนผู้ชนะ**']
    L += ['', 'ต่อ fold (ดู `fold_comparison.csv`):', '']
    L += md_table(FOLDCMP[FOLDCMP['unit'].eq(u)], ['fold', 'method_id', 'n', 'mean_poisson_deviance', 'mae', 'beats_strongest_baseline'],
                  ['fold', 'วิธี', 'n', 'deviance', 'MAE', 'ชนะ baseline'])
    # feature additions: every pair, benefit and harm
    a = ABL[ABL['unit'].eq(u) & ABL['status'].eq('ok')].sort_values(['config_id', 'comparison_source', 'variant', 'reference'])
    inc = ABL[ABL['unit'].eq(u) & ABL['status'].ne('ok')]
    L += ['', f'### การเพิ่ม feature ({u}; keys เดียวกันทุกคู่; ลบ = ดีขึ้น, บวก = แย่ลง; แสดงทุกคู่ ไม่คัดเลือก)', '',
          f"- benefit {int(a['effect'].eq('benefit').sum())} คู่, harm {int(a['effect'].eq('harm').sum())} คู่, no_change {int(a['effect'].eq('no_change').sum())} คู่; "
          f"ช่วง bootstrap คร่อม 0 จำนวน {int(a['interval'].eq('includes_0').sum())}/{len(a)} คู่ (แยกไม่ออกจากศูนย์); ไม่ครบ {len(inc)} คู่", '']
    L += md_table(a, ['config_id', 'variant', 'reference', 'comparison_source', 'delta_deviance', 'relative_change', 'delta_mae', 'folds_improved', 'boot_lower',
                      'boot_upper', 'effect'],
                  ['config', 'variant', 'เทียบกับ', 'ที่มา', 'Δ deviance', 'relative', 'Δ MAE', 'fold ดีขึ้น', 'boot ล่าง', 'boot บน', 'ผล'])
    sens = ABL_SENS[ABL_SENS['unit'].eq(u) & ABL_SENS['status'].eq('ok')]
    L += ['', f"- ความไวต่อความครอบคลุมของแหล่งข้อมูล ({len(sens)} แถว, `ablation_coverage_sensitivity.csv`): เทียบคู่เดิมเฉพาะเซลล์ที่มี/ไม่มี weather หรือ traffic "
          '(สองวิธีใช้ keys เดียวกันเสมอ) — ความครอบคลุมเป็นผลของการเก็บข้อมูลย้อนหลังและสัมพันธ์กับปี จึง **ไม่ใช่หลักฐานว่า feature มีผล**', '']
    # slices of the recommended method, per period, compact dimensions (full tables in CSV)
    rm = dj['recommended_method']
    L += [f'### Error slice ของ `{rm}` ({u}) — ไฟล์เต็ม `slices_{u}.csv` (เดือน/จังหวัด/ทุกวิธีที่สนใจ)', '',
          f'กลุ่มที่มีน้อยกว่า {MIN_CELLS} เซลล์หรือน้อยกว่า {MIN_EVENTS} เหตุการณ์ถูกระบุว่า "ข้อมูลน้อย"; กลุ่มว่างแสดง n = 0; '
          f'กลุ่มตามผลลัพธ์ (0 เหตุการณ์ / high count >= {int(HIGH_THR[u])} จาก OOF quantile {HIGH_Q}) เป็นการวินิจฉัยย้อนหลัง ไม่ใช่ตัวทำนายหรือสาเหตุ; '
          'สถานะ missing ของ weather/traffic/bank calendar ใช้เพื่อวินิจฉัยเท่านั้น ไม่ใช่ input และไม่ใช่ข้ออ้างเชิงสาเหตุ', '']
    dims = ['weekday', 'festival', 'bank_calendar', 'weather', 'traffic', 'outcome_bin', 'high_count'] + (['time_block'] if u == '6h' else []) + ['year']
    for per in PERIOD_FRAMES[u]:
        t = SLICES[SLICES['unit'].eq(u) & SLICES['period'].eq(per) & SLICES['method_id'].eq(rm) & SLICES['dimension'].isin(dims)]
        L += [f'**{PERIOD_LABEL[per]}**', '']
        L += md_table(t, ['dimension', 'level', 'n', 'sum_y', 'sum_pred', 'bias', 'mae', 'rmse', 'mean_poisson_deviance', 'support'],
                      ['มิติ', 'กลุ่ม', 'n', 'จริง', 'ทำนาย', 'bias', 'MAE', 'RMSE', 'deviance', 'support'])
        L += ['']
    L += [f'### Calibration และ residual ({u})', '',
          f'- `calibration_bins_{u}.csv`: ขอบ bin จาก quantile ของ prediction OOF เท่านั้น (ค่าซ้ำรวม bin, bin ว่างคงไว้) แล้วใช้ขอบเดิมกับ 2024/2025',
          f'- `largest_residuals_{u}.csv`: residual รวมสูงสุด (ทำนาย − จริง) ตาม จังหวัด×เดือน, วันที่ และจังหวัด ของแต่ละช่วงแยกกัน', '']
    cb = CALIB[CALIB['unit'].eq(u) & CALIB['method_id'].eq(rm)]
    L += md_table(cb, ['period', 'bin', 'edge_lo', 'edge_hi', 'n', 'mean_pred', 'mean_y', 'obs_over_pred', 'support'],
                  ['ช่วง', 'bin', 'ขอบล่าง', 'ขอบบน', 'n', 'ทำนายเฉลี่ย', 'จริงเฉลี่ย', 'จริง/ทำนาย', 'support'])
    rr = RESID[RESID['unit'].eq(u) & RESID['method_id'].eq(rm) & RESID['grouping'].eq('province_key+sl_year_month')].groupby('period', sort=False).head(5)
    L += ['', f'residual รวมสูงสุด 5 อันดับ (จังหวัด×เดือน) ของ `{rm}`:', '']
    L += md_table(rr, ['period', 'province_key', 'year_month', 'n', 'sum_y', 'sum_pred', 'residual_total', 'support'],
                  ['ช่วง', 'จังหวัด', 'เดือน', 'n', 'จริง', 'ทำนาย', 'residual', 'support'])
    L += ['']
L += ['## 3. Drift ปีหลังและการเปลี่ยนแปลงความครอบคลุมของแหล่งข้อมูล', '']
L += md_table(DRIFT, ['unit', 'period', 'n_eligible_cells', 'n_days', 'first_date', 'last_date', 'mean_y', 'share_zero', 'weather_available_share',
                      'traffic_available_share', 'bank_known_share', 'recommended_bias', 'recommended_sum_pred_over_sum_y'],
              ['หน่วย', 'ช่วง', 'เซลล์ eligible', 'วัน', 'เริ่ม', 'สิ้นสุด', 'y เฉลี่ย', 'สัดส่วน 0', 'weather มี', 'traffic มี', 'bank รู้', 'bias ที่แนะนำ', 'ทำนาย/จริง'])
L += ['']
for u in UNITS:
    cf = COV_FACTS[u]
    L.append(f"- {u}: เซลล์ panel/eligible ต่อ split = {cf['cells_by_split']}; province-year ที่มี weather อย่างน้อยบางส่วน "
             f"{cf['province_years_with_any_weather']}/{cf['province_years_total']} (รายปี {cf['weather_province_years_by_year']})")
L += ['- ป้ายปี 2024 ถูกตัดเป็น eligible เพียงส่วนน้อยของเซลล์ (ตัวกรองนี้ได้จากผลลัพธ์ย้อนหลัง ใช้ล่วงหน้าไม่ได้) ผลปี 2024 จึงแทนเฉพาะวันที่ป้ายทราบ ไม่ใช่ทั้งปี',
      '- 2024/2025 ไม่มีข้อมูลอากาศ: ทุกแถวปีหลังเป็น weather missing ดังนั้นความต่างระหว่าง variant ที่มี/ไม่มี weather ในปีหลังสะท้อนการเติมค่าที่หายไป ไม่ใช่ผลของอากาศ',
      '- ความครอบคลุมรายเดือน: `coverage_by_month_<unit>.csv`, กราฟ `fig_coverage_<unit>.png`', '',
      '## 4. ข้อจำกัด', '',
      '- 2024 = development ที่เคยเห็นแล้ว, 2025 = reused/previously-seen: ไม่มีชุดใดเป็น untouched test; ห้ามเลือกใหม่จากผลปีหลัง',
      '- bootstrap เป็นการสำรวจหลังการเลือก ไม่ปรับ multiple comparison; slice/กลุ่มย่อยเป็นการบรรยาย ไม่ใช่การประมาณผลเชิงสาเหตุ',
      '- เวลาเผยแพร่จริงของรายงานอุบัติเหตุและ weather ย้อนหลังยังไม่ยืนยัน; traffic เป็นค่าเฉลี่ยจุดสำรวจ ไม่ใช่ exposure',
      '- ผลทั้งหมดไม่อนุญาตการพยากรณ์เชิงปฏิบัติการ, เว็บที่ deploy, หรือการประเมินความเสี่ยงรายบุคคล/เส้นทาง', '',
      '## 5. ไฟล์', '',
      '- ตาราง: `decision_summary.csv`, `gate_checks.csv`, `fold_comparison.csv`, `ablation_all_pairs.csv`, `ablation_coverage_sensitivity.csv`, '
      '`slices_<unit>.csv`, `calibration_bins_<unit>.csv`, `largest_residuals_<unit>.csv`, `weekly_actual_vs_predicted_<unit>.csv`, `coverage_by_month_<unit>.csv`, '
      '`drift_summary.csv`, `metric_reconciliation.csv`, `ablation_reconciliation.csv`',
      '- ผล notebook 11 (อ่านอย่างเดียว): `../models/oof_metrics_pooled.csv`, `../models/oof_metrics_by_fold.csv`, `../models/fold_boundaries.csv`, '
      '`../models/diagnostic_metrics.csv`, `../models/selection_lock.json`',
      '- กราฟ: ' + ', '.join(f'`{n}`' for n in FIGS), '- แถวรายเซลล์ (ไม่อยู่ใน git): `data/v2/analysis/analysis_rows_<unit>.csv.gz`']
p = REP / 'analysis_report_th.md'
p.write_text('\n'.join(L) + '\n', 'utf-8')
WRITTEN.append(p)

DM = {u: DMET12[u].set_index(['split', 'method_id']) for u in UNITS}
C = ['# Model card V2 — มุมมองหลังวิเคราะห์ (Notebook 12, ร่างงานวิจัย)', '',
     '- ใช้งาน: วิจัยเท่านั้น; production_ready = false; operational BLOCKED; ไม่ใช่ความเสี่ยงรายบุคคล/เส้นทาง, ไม่ใช่อุบัติเหตุทุกถนน, ไม่ใช่เว็บที่ deploy',
     '- ข้อมูล: panel V2 ที่ตรวจรับแล้ว (notebook 10); ตัวเลือกล็อกใน `../models/selection_lock.json` ก่อนดูปีหลัง; 12 ไม่ได้เปลี่ยนตัวเลือก', '']
for u in UNITS:
    dj = DECISION_JSON[u]
    C += [f'## {u}', f"- วิธีที่แนะนำ: `{dj['recommended_method']}` ({dj['recommendation_kind']}); gate ผ่าน = {dj['gate_passed']}",
          f"- best candidate: `{dj['best_candidate']}`; strongest baseline: `{dj['strongest_baseline']}`", f"- เหตุผล: {'; '.join(dj['reasons'])}", '',
          '| วิธี | ช่วง | n | deviance | MAE | bias |', '|---|---|---|---|---|---|']
    for m in [x for x in dict.fromkeys([dj['recommended_method'], dj['best_candidate'], dj['strongest_baseline']]) if x]:
        r = POOLED12[u].set_index('method_id').loc[m]
        C.append(f"| {m} | pooled OOF (เลือก) | {_f(r['n'])} | {_f(r['mean_poisson_deviance'])} | {_f(r['mae'])} | {_f(r['bias'])} |")
        for sp in DIAG_SPLITS:
            if (sp, m) in DM[u].index:
                r = DM[u].loc[(sp, m)]
                C.append(f"| {m} | {sp} (เคยเห็นแล้ว) | {_f(r['n'])} | {_f(r['mean_poisson_deviance'])} | {_f(r['mae'])} | {_f(r['bias'])} |")
    C.append('')
C += ['## ข้อควรระวัง', '- ไม่มี untouched test; 2024/2025 เป็น diagnostic ที่เคยเห็นแล้ว', '- ตัวกรอง eligible ได้จากผลลัพธ์; ป้ายปี 2024 ทราบเพียงบางส่วน',
      '- feature ที่ไม่ช่วย/ทำให้แย่ลงแสดงไว้ใน `analysis_report_th.md` และ `ablation_all_pairs.csv`', '- รายละเอียด: `analysis_report_th.md`']
p = REP / 'model_card_v2_analysis_th.md'
p.write_text('\n'.join(C) + '\n', 'utf-8')
WRITTEN.append(p)

FN12_SHA = sha256_bytes(functions_cells_source(NB12).encode('utf-8')) if NB12.is_file() else None
AMAN = {
    'schema': ANALYSIS_SCHEMA, 'producer': ANALYSIS_PRODUCER, 'status': 'completed', 'status_note': 'research analysis; passing checks is not acceptance',
    'production_ready': False, 'operational_status': 'BLOCKED', 'started_utc': T_RUN0.replace(microsecond=0).isoformat(), 'finished_utc': utc_now(),
    'functions_cells_sha256': FN12_SHA, 'code_cells_sha256': code_cells_sha256(NB12) if NB12.is_file() else None,
    'versions': VERSIONS, 'tolerance': TOL,
    'parameters': {'min_cells': MIN_CELLS, 'min_events': MIN_EVENTS, 'n_cal_bins': N_CAL_BINS, 'top_residuals': TOP_RESIDUALS, 'high_q': HIGH_Q,
                   'high_count_threshold': HIGH_THR},
    'inputs': [{'path': k, 'sha256': v} for k, v in sorted(SNAP_BEFORE.items())],
    'notebook11': {'model_manifest_sha256': sha256_file(MAN_PATH), 'functions_cells_sha256': FN11_SHA, 'selection_lock_sha256': IT['selection_lock']['sha256'],
                   'runner_log': RUNNER11, 'output_root': str(OUT11)},
    'experiment_lock_sha256': ACTUAL_LOCK_HASHES['experiment_lock_sha256'], 'model_config_sha256': ACTUAL_LOCK_HASHES['model_config_sha256'],
    'roots': {'model_reports': str(MODEL_REP), 'model_data': str(MODEL_DATA), 'prep_data': str(PREP_DATA), 'reports': str(REP), 'analysis_data': str(ADATA)},
    'decision': DECISION_JSON, 'coverage_facts': COV_FACTS,
    'checks_run': CHECKS,
    'checks_not_run_by_notebook': ['visual inspection of the PNG figures (manual)', 'independent acceptance review'],
    'caveats': ['no untouched test: 2024 development and 2025 reused diagnostics were seen before',
                'bootstrap intervals are exploratory, after selection, not adjusted for multiple comparisons',
                'slices and coverage subsets are descriptive; small/empty slices are labelled and not effect estimates',
                'missingness / availability labels are diagnostics only, never model inputs or causal evidence',
                'eligible-label mask is outcome-derived (offline only); 2024 labels are only partly known',
                'no production, deployed-web, all-road or personal/route risk claim'],
    'outputs': [{'path': str(Path(p).relative_to(OUT)) if inside(p, OUT) else str(p), 'sha256': sha256_file(p)} for p in WRITTEN],
}
write_json(REP / 'analysis_manifest.json', AMAN)
print('analysis report ->', REP / 'analysis_report_th.md', '| outputs', len(WRITTEN))

In [ ]:
# Final checks: read-only inputs unchanged, both locks still verify, strict JSON, every output hash equals the manifest, research-only flags
SNAP_AFTER = snapshot(READ_ONLY_FILES)
changed = snapshot_diff(SNAP_BEFORE, SNAP_AFTER)
assert not changed, f'read-only inputs changed during notebook 12: {changed[:5]}'
verify_experiment_lock(LOCK_PATH, PROTOCOL_PATH, CFG_PATH, PM_PATH, PM, IC['experiment_lock_schema'], IC['experiment_lock_fields'])
verify_selection_lock(SEL_LOCK_PATH, IT['selection_lock']['sha256'], ACTUAL_LOCK_HASHES, OUT11)
for p in sorted(REP.glob('*.json')):
    read_json_strict(p)
M12 = read_json_strict(REP / 'analysis_manifest.json')
for o in M12['outputs']:
    fp = Path(o['path']) if Path(o['path']).is_absolute() else OUT / o['path']
    assert sha256_file(fp) == o['sha256'], f"output changed after the manifest: {o['path']}"
assert M12['production_ready'] is False and M12['operational_status'] == 'BLOCKED'
assert all(c['status'] == 'passed' for c in M12['checks_run'])
print(f'final checks passed: {len(SNAP_AFTER)} read-only files unchanged, experiment + selection locks verified, strict JSON, '
      f"{len(M12['outputs'])} output hashes verified; production_ready false")

## ข้อจำกัด

- ไม่มี untouched test; ผลปีหลังเป็น diagnostic ที่เคยเห็นแล้ว
- bootstrap/slice เป็นการสำรวจ ไม่ปรับ multiple comparison และไม่ใช่การประมาณผลเชิงสาเหตุ
- ผลใด ๆ ในที่นี้ไม่อนุญาตการพยากรณ์เชิงปฏิบัติการหรือเว็บที่ deploy